# PKB monitoring events in soft collection: impact and Information Value

**Question.** Which credit-bureau (PKB) monitoring events carry signal for soft collection, and how much do they
add on top of the in-house data the collection strategy already uses?

**Inputs** (the same objects as in the working notebook)

| object | grain | columns used |
|---|---|---|
| `ovrd_contr` | one row per contract and `act_date` with overdue (`ovd_days_cnt` > 0), merged with contract attributes | `contract_number`, `iin`, `act_date`, `ovd_days_cnt`, `process_ref`, `open_date`, `close_date`, `end_date`, `status_name`, `contract_amount`, `partner_name`, `product_id` |
| `hes` | one row per PKB monitoring event | `bin_iin`, `rule_name`, `old_value`, `new_value` (JSON text), `created_date`, `event_date`, `internal_event_id` |
| `contracts` *(optional)* | one row per contract issued 2025-01-01 … 2026-10-15 | contract attributes; switches on the early-warning section (part B) |

**Method (point in time, no look-ahead)**
1. **Delinquency episodes** are rebuilt from DPD rows: `dlq_start = act_date − (ovd_days_cnt − 1)`. This works whether
   `ovrd_contr` holds a row for every overdue day or only for some days of the month.
2. **Population (part A)**: every entry of a contract into soft collection (scored on the day it reaches `OBS_DPD`,
   default DPD 1) whose scoring day falls inside the monitoring coverage and whose outcome is fully observed.
3. **Target**: `bad = 1` if the episode reaches DPD ≥ 31 (roll from soft into hard collection).
   Secondary target: not cured by DPD 16.
4. **Features**: PKB events that reached the bank **before** the scoring day (`created_date` < score date),
   aggregated over 7 / 14 / 30 days per rule, plus amounts and changes parsed from the JSON. Events on the bank's own
   contracts (`contract_code` equal to our `contract_number` or `process_ref`) are kept in a separate group,
   because they repeat information the bank already has.
5. **Analysis**: WoE/IV with monotonic binning, Gini and KS per feature, event lift per rule, weekly stability and PSI,
   correlation filter, and the Gini a model gains over an in-house baseline on an out-of-time sample (bootstrap CI).
6. **Part B (optional)**: the same features for contracts that are still current, predicting entry into
   delinquency within 30 days (early warning before soft collection starts).

**How to run on the real data**: run these cells in the kernel where `ovrd_contr` and `hes` (and `contracts`, if
you have it) are already loaded, or paste the cells into that notebook. `DEMO_MODE = False` is the default; the names
of your dataframes are set in `OVRD_VAR`, `HES_VAR`, `CONTRACTS_VAR`. Section 2b validates the dataframes before any
heavy step and stops with a clear message if something does not fit.
With `DEMO_MODE = True` the notebook builds a synthetic data set with the same columns and JSON layouts,
so every cell can be checked end to end. **Synthetic results say nothing about the real portfolio.**

In [ ]:
import os, re, gc, json, math, time, uuid, warnings
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

try:                                   # 3-5x faster JSON parsing when available
    import orjson
    _json_loads = orjson.loads
except ImportError:
    _json_loads = json.loads

try:
    display                            # noqa: F821  (exists inside Jupyter)
except NameError:
    display = print

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

# ------------------------------------------------------------------ run mode
DEMO_MODE = False           # False: your dataframes already loaded in this kernel. True: synthetic demo data.
SEED = 42

# ------------------------------------------------------------------ names of YOUR dataframes in this kernel
OVRD_VAR = 'ovrd_contr'     # overdue rows merged with contracts
HES_VAR = 'hes'             # PKB monitoring events
CONTRACTS_VAR = 'contracts' # optional: all issued contracts (enables part B); a missing name simply skips part B

# ------------------------------------------------------------------ column names (change if yours differ)
C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD = 'contract_number', 'iin', 'act_date', 'ovd_days_cnt'
C_PROCESS_REF, C_OPEN, C_CLOSE, C_END = 'process_ref', 'open_date', 'close_date', 'end_date'
C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT = 'status_name', 'contract_amount', 'partner_name', 'product_id'
H_IIN, H_RULE, H_OLD, H_NEW = 'bin_iin', 'rule_name', 'old_value', 'new_value'
H_ID, H_EVENT_DATE, H_CREATED = 'internal_event_id', 'event_date', 'created_date'

# ------------------------------------------------------------------ point-in-time logic
EVENT_TIME_COL = H_CREATED  # when the event reached the bank (safe). event_date = when it happened at PKB.
EVENT_LAG_DAYS = 0          # extra days before an event may be used (production latency buffer)
OBS_DPD = 1                 # scoring point: the day the episode reaches this DPD (1 = first day of soft collection)
BAD_DPD = 31                # main target: the episode reaches DPD >= 31
EXTRA_TARGETS = {'nocure_16': 16}   # secondary targets: name -> DPD threshold (add 'roll_61': 61 once history allows)
WINDOWS = (7, 14, 30)       # look-back windows in days
IMPACT_WINDOW = 14          # window used for the rule-level lift table
MIN_HISTORY_DAYS = 14       # first scoring day = monitoring start + this many days
HES_START = None            # monitoring start; None = detect from daily volumes, or a date such as '2026-09-07'
DATA_END = None             # last day with complete DPD data; None = max(act_date)
EPISODE_START_TOL_DAYS = 1  # jitter of dlq_start tolerated inside one episode
BANK_RULES = ('partial_ovd', 'refin_out_80')   # PKB rules defined relative to our own bank

# ------------------------------------------------------------------ binning, selection, model
MAX_BINS = 6
MIN_BIN_SHARE = 0.05
MIN_BADS_RELIABLE = 100     # IV resting on fewer bad cases outside the main bin is flagged
IV_SELECT = 0.02
CORR_MAX = 0.75
MAX_MODEL_FEATURES = 15
TRAIN_SHARE = 0.70          # out-of-time split by scoring day
N_BOOT = 200
RUN_HGB = True              # gradient boosting cross-check of the incremental value
RUN_HES_DQ = True           # data-quality pass over the whole hes table (weekly counts per rule, delays)
DQ_SAMPLE = 2_000_000       # rows sampled from hes for the delay statistics and the preflight checks
STOP_ON_PREFLIGHT_FAIL = True

# ------------------------------------------------------------------ part B: early warning (needs `contracts`)
RUN_EWS = True
EWS_HORIZON = 30            # a current contract enters delinquency within this many days ...
EWS_MIN_DPD = 1             # ... and that episode reaches at least this DPD
EWS_SNAPSHOT_FREQ_DAYS = 7
EWS_MAX_ROWS = 400_000

OUTPUT_DIR = 'pkb_iv_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# chart style: thin marks, hairline grid, fixed categorical order
PAL = {'blue': '#2a78d6', 'orange': '#eb6834', 'aqua': '#1baf7a', 'yellow': '#eda100',
       'magenta': '#e87ba4', 'green': '#008300', 'violet': '#6250d6', 'red': '#e34948',
       'gray': '#8c8b85', 'ink': '#0b0b0b', 'muted': '#52514e'}
SERIES = [PAL[k] for k in ('blue', 'orange', 'aqua', 'yellow', 'magenta', 'green', 'violet', 'red')]
GROUP_COLOR = {'pkb_other': PAL['blue'], 'pkb_bank_rule': PAL['orange'], 'pkb_own': PAL['aqua'], 'baseline': PAL['yellow']}
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae', 'axes.linewidth': 0.6,
    'axes.titlesize': 10.5, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'axes.labelcolor': PAL['muted'],
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'xtick.color': PAL['muted'], 'ytick.color': PAL['muted'],
    'legend.fontsize': 8, 'legend.frameon': False, 'lines.linewidth': 2})
print('pandas', pd.__version__, '| numpy', np.__version__, '| json:', _json_loads.__module__)

## 1. Helpers
Normalisation of keys (IIN with leading zeros, contract codes), PKB rule catalogue, timers.

In [ ]:
class Timer:
    def __init__(self, msg):
        self.msg = msg
    def __enter__(self):
        self.t0 = time.time()
        print(f'> {self.msg} ...', flush=True)
        return self
    def __exit__(self, *exc):
        print(f'  done in {time.time() - self.t0:,.1f}s', flush=True)


_NULLS = ['', 'None', 'none', 'nan', 'NaN', 'NAN', 'NONE', 'NULL', 'null', '<NA>', 'NaT']


def norm_iin(s):
    """IIN/BIN as a 12-character string; restores leading zeros lost by int/float conversion."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    short_digits = s.str.fullmatch(r'\d{1,11}').fillna(False).astype(bool)
    s = s.where(~short_digits, s.str.zfill(12))
    return s.where(~s.isin(_NULLS), np.nan).astype(object)


def norm_code(s):
    """Contract code as an upper-case trimmed string (same rule for our contracts and PKB contract_code)."""
    s = pd.Series(s, copy=False).astype(str).str.strip().str.upper()
    return s.where(~s.isin([x.upper() for x in _NULLS]), np.nan).astype(object)


_PD2 = int(pd.__version__.split('.')[0]) >= 2


def to_dt(s):
    """Robust date parsing. pandas >= 2 infers ONE format from the first value and turns other layouts into NaT,
    so ISO8601 is requested explicitly (covers 'YYYY-MM-DD', with time, with fractions, date/datetime objects)."""
    s = pd.Series(s, copy=False) if not isinstance(s, pd.Series) else s
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    if not _PD2:
        return pd.to_datetime(s, errors='coerce')
    out = pd.to_datetime(s, errors='coerce', format='ISO8601')
    lost = out.isna() & s.notna()
    if lost.mean() > 0.01:                       # non-ISO layout: parse value by value
        out = pd.to_datetime(s, errors='coerce', format='mixed')
    return out


def days(n):
    return pd.Timedelta(days=int(n))


# PKB monitoring rules: (lower-case fragment of rule_name, code, family, description).
# Order matters: the first fragment found in the rule name wins.
RULES = [
    ('кредитной заявки', 'app_online', 'credit_hunger', 'Online credit application'),
    ('пролонгация', 'prolong', 'relief', 'Loan prolongation'),
    ('любой поставщик', 'lookup_other', 'credit_hunger', 'Another provider (not PKB) looked up the client'),
    ('сумма просрочки', 'ovd_sum_chg', 'delinquency', 'Total overdue amount went up/down'),
    ('с целью рефинанс', 'new_loan_refin', 'credit_hunger', 'New loan for refinancing'),
    ('появление нового займа', 'new_loan', 'credit_hunger', 'New loan appeared'),
    ('урегулирование', 'settlement', 'positive', 'Debt settlement'),
    ('лудоман', 'st_gambler', 'status', 'Status «Лудоман» (gambling self-ban)'),
    ('отписка', 'unsubscribe', 'technical', 'Unsubscribed: no active contracts (technical)'),
    ('призывник', 'st_recruit', 'status', 'Status «Призывник» (conscript)'),
    ('залогу', 'collateral_chg', 'other', 'Collateral change'),
    ('запросов на кредитный отчет', 'report_req', 'credit_hunger', 'Credit report request'),
    ('новый контракт', 'new_contract', 'credit_hunger', 'New contract appeared'),
    ('реструктур', 'restruct', 'relief', 'Loan restructured / refinanced / revised'),
    ('стоп-кредит', 'st_stopcredit', 'status', 'Status «Стоп-Кредит» (self-imposed credit ban)'),
    ('статуса по кредиту изменился', 'debt_terminated', 'delinquency', 'Debt terminated without full repayment'),
    ('частичная просрочка', 'partial_ovd', 'bank_rule', 'Overdue with us while paying other lenders'),
    ('банкрот', 'st_bankrupt', 'status', 'Status «Банкротство»'),
    ('переуступка', 'loan_assigned', 'delinquency', 'Loan closed by assignment (sold on)'),
    ('закрытие займа', 'loan_closed', 'positive', 'Loan closed (repaid)'),
    ('сделать сравнение', 'refin_out_80', 'bank_rule', '>=80% of our principal repaid after a new loan elsewhere'),
    ('ежемесячного платежа', 'instalment_chg', 'debt_burden', 'Total monthly payment changed'),
    ('дней просрочки', 'dpd_chg', 'delinquency', 'Days past due changed'),
    ('основного долга', 'debt_chg', 'debt_burden', 'Total principal went up/down'),
]
FAMILY = {code: fam for _, code, fam, _ in RULES}
RULE_DESC = {code: desc for _, code, _, desc in RULES}
STATUS_RULES = [code for _, code, fam, _ in RULES if fam == 'status']


def map_rule_names(names):
    """{rule_name: code}. Unknown rule names get an automatic code so they still produce count features."""
    out = {}
    for nm in names:
        low = str(nm).lower()
        for frag, code, _, _ in RULES:
            if frag in low:
                out[nm] = code
                break
        else:
            out[nm] = 'other_' + re.sub(r'\W+', '_', low).strip('_')[:30]
            FAMILY.setdefault(out[nm], 'unknown')
            RULE_DESC.setdefault(out[nm], str(nm)[:80])
    return out


def iv_strength(iv):
    if not np.isfinite(iv):
        return ''
    return ('not useful' if iv < 0.02 else 'weak' if iv < 0.1 else 'medium' if iv < 0.3
            else 'strong' if iv < 0.5 else 'suspicious: check leakage')

# JSON helpers (used by the preflight checks and by the event parser in section 7)
NUM_KEYS = ('overdue_sum', 'debt_sum', 'instalment', 'overdue_instalment_count')
NAN = float('nan')
_EMPTY_NEW = (None, NAN, NAN, NAN, NAN, NAN, NAN, NAN)
_EMPTY_OLD = (NAN, NAN)


def _num(v):
    if v is None:
        return NAN
    if isinstance(v, bool):
        return 1.0 if v else 0.0
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        t = v.strip().lower()
        if t in ('true', 'false'):
            return 1.0 if t == 'true' else 0.0
        try:
            return float(t.replace(' ', '').replace(',', '.'))
        except ValueError:
            return NAN
    return NAN


def _id(v):
    return _num(v.get('id')) if isinstance(v, dict) else _num(v)


def _loads(s):
    if isinstance(s, dict):
        return s
    if not isinstance(s, (str, bytes)) or len(s) < 2:
        return None
    try:
        d = _json_loads(s)
    except Exception:
        return None
    return d if isinstance(d, dict) else None


def _flag(d):
    for k, v in d.items():
        if k.startswith(('is_', 'has_')) or k == 'stopcredit':
            x = _num(v)
            if x == x:
                return x
    return NAN


def _first_num(d):
    for k in NUM_KEYS:
        v = d.get(k)
        if v is not None:
            return _num(v)
    return NAN


def _parse_new(s):
    d = _loads(s)
    if d is None:
        return _EMPTY_NEW
    code = d.get('contract_code')
    code = str(code).strip().upper() if code not in (None, '') else None
    pdays = NAN
    a, b = d.get('prolongation_start_date'), d.get('prolongation_end_date')
    if a and b:
        try:
            pdays = float((date.fromisoformat(str(b)[:10]) - date.fromisoformat(str(a)[:10])).days)
        except ValueError:
            pass
    return (code, _id(d.get('creditor_type')), _first_num(d), _flag(d), _num(d.get('summa_zaima')),
            _num(d.get('zadolzhenost')), _num(d.get('outstanding_debt')), pdays)


def _parse_old(s):
    d = _loads(s)
    if d is None:
        return _EMPTY_OLD
    return (_first_num(d), _flag(d))

## 2. Data
`DEMO_MODE = True` generates a synthetic portfolio with the same columns, dtypes and JSON layouts as `ovrd_contr`,
`hes` and `contracts` (about 60k clients, 9 weeks of monitoring). The generator plants a known signal so that
every step of the analysis can be checked.

With `DEMO_MODE = False` the notebook uses the objects already in memory. Alternatives are shown in the cell.

In [ ]:
CT = {1: {'id': '1', 'name': 'Банк второго уровня'}, 2: {'id': '2', 'name': 'Микрофинансовая организация'}}
SYN_RULE_NAMES = {   # rule_name texts used by the generator (real names are matched by fragments, see RULES)
    'app_online': 'Мониторинг кредитной заявки (онлайн заявки)',
    'prolong': 'Пролонгация по займу',
    'lookup_other': 'Любой поставщик (кроме ПКБ) искал информацию по данному субъекту',
    'ovd_sum_chg': 'Совокупная сумма просрочки увеличилась/уменьшилась',
    'new_loan': 'Мониторинг появление нового займа',
    'settlement': 'Мониторинг факта урегулирование задолженности',
    'st_gambler': 'Мониторинг статуса «Лудоман»',
    'unsubscribe': 'Отписка субъектов без активных контрактов',
    'st_recruit': 'Мониторинг статуса «Призывник»',
    'collateral_chg': 'Изменения по залогу займа',
    'report_req': 'Мониторинг запросов на кредитный отчет',
    'new_contract': 'У субъекта появился новый контракт',
    'restruct': 'Мониторинг изменения по кредиту - (Реструктурирован/Рефинансирован/Пересмотрен)',
    'new_loan_refin': 'Появление нового займа с целью Рефинансирование, где получатель единственный кредитор',
    'st_stopcredit': 'Мониторинг статуса «Стоп-Кредит»',
    'debt_terminated': 'Мониторинг статуса по кредиту изменился – Ссудная задолженность прекращена без полного исполнения обязательства',
    'partial_ovd': 'Частичная просрочка по кредиторам (заемщик допустил просрочку в организации получателя, но продолжает платить в других)',
    'st_bankrupt': 'Мониторинг статуса «Банкротства»',
    'loan_closed': 'Мониторинг закрытие займа (погашение)',
    'loan_assigned': 'Мониторинг закрытие займа – переуступка',
    'refin_out_80': 'Сделать сравнение, у кого до погашения не менее 80% от ОД займа – появился кредит в другом банке',
    'instalment_chg': 'Совокупная сумма ежемесячного платежа изменилась',
    'dpd_chg': 'Мониторинг изменения дней просрочки займа',
    'debt_chg': 'Совокупная сумма основного долга увеличилась/уменьшилась',
}


def _j(d):
    return json.dumps(d, ensure_ascii=False)


def _ext_codes(r, n):
    """Contract codes of other lenders, in several formats."""
    k = r.integers(0, 5, n)
    x = r.integers(10**6, 10**9, n)
    fmt = ['KZ{:010d}', 'CO2026{:012d}', 'Z{:08d}A0219-3', 'КАРТ-{:010d}', '{:d}']
    return [fmt[a].format(b) for a, b in zip(k, x)]


def _payloads(code, sv, r, codes=None, dpd_old=None, dpd_new=None):
    """old_value / new_value JSON strings for n events of one rule (layouts copied from the real table)."""
    n = len(sv)
    mfo = r.random(n) < 1 / (1 + np.exp(-(-0.9 + 0.9 * sv)))
    ct = [CT[2] if m else CT[1] for m in mfo]
    cc = codes if codes is not None else _ext_codes(r, n)
    amt = np.round(np.exp(r.normal(12.2 + 0.25 * sv, 0.8)), -3)
    none = [None] * n
    if code == 'app_online':
        new = [_j({'object': {'id': None, 'name': None}, 'purpose': {'id': '04', 'name': 'Потребительские цели'},
                   'summa_zaima': str(int(a)), 'credit_app_id': int(i), 'creditor_type': c})
               for a, i, c in zip(amt, r.integers(10**8, 10**9, n), ct)]
        return none, new
    if code in ('prolong', 'new_loan_refin', 'loan_closed'):
        debt = np.round(amt * r.uniform(0.2, 0.95, n), 0)
        new = []
        for a, d_, c, k, p in zip(amt, debt, ct, cc, r.integers(0, 10, n)):
            d = {'object': {'id': '10', 'name': 'Прочие'},
                 'purpose': {'id': '16', 'name': 'Рефинансирование'} if code == 'new_loan_refin' else {'id': '09', 'name': 'Прочие'},
                 'summa_zaima': str(int(a)) if code != 'loan_closed' else float(a), 'contract_code': k, 'creditor_type': c}
            if code != 'loan_closed':
                d['zadolzhenost'] = str(int(d_))
            if code == 'prolong':
                d['prolongation_start_date'] = '2026-09-08T00:00:00+05:00'
                d['prolongation_end_date'] = f'2026-09-{14 + int(p):02d}T00:00:00+05:00'
            if code == 'new_loan_refin':
                d.update({'platezh': None, 'valuta_platezha': None})
            new.append(_j(d))
        return none, new
    if code == 'lookup_other':
        return none, [_j({'report': {'id': '6', 'name': 'Кредитный отчет – расширенный физическое лицо'}})] * n
    if code == 'report_req':
        return none, [_j({'creditor_type': c, 'credit_report_id': str(i)}) for c, i in zip(ct, r.integers(10**8, 10**9, n))]
    if code == 'ovd_sum_chg':
        o = np.round(np.exp(r.normal(10 + 0.4 * sv, 1.0)), 2) * (r.random(n) < 0.7)
        up = r.random(n) < 1 / (1 + np.exp(-0.9 * sv))
        nv = np.where(up, o + np.round(np.exp(r.normal(9.5, 0.8, n)), 2), o * r.uniform(0, 0.6, n))
        old = [_j({'overdue_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'overdue_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code in ('new_loan', 'restruct', 'collateral_chg', 'debt_terminated', 'loan_assigned'):
        return none, [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': None}) for k, c in zip(cc, ct)]
    if code == 'new_contract':
        old = [_j({'end_date': None, 'start_date': None, 'subject_role': {'id': None, 'name': None},
                   'contract_code': None, 'financial_institution': {'id': 0, 'name': None}})] * n
        new = [_j({'end_date': '2028-09-01', 'start_date': '2026-09-01', 'subject_role': {'id': 1, 'name': 'Заемщик'},
                   'contract_code': k, 'financial_institution': {'id': int(f), 'name': 'ФИ'}})
               for k, f in zip(cc, r.integers(1, 400, n))]
        return old, new
    if code == 'debt_chg':
        o = np.round(np.exp(r.normal(13, 1.0, n)), 2)
        up = r.random(n) < 1 / (1 + np.exp(-(-1.6 + 0.6 * sv)))
        nv = np.where(up, o * r.uniform(1.0, 1.3, n), o * r.uniform(0.9, 1.0, n))
        old = [_j({'debt_sum': float(a), 'contract_code': k, 'creditor_type': c}) for a, k, c in zip(o, cc, ct)]
        new = [_j({'debt_sum': float(round(b, 2)), 'contract_code': k, 'creditor_type': c}) for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'instalment_chg':
        o = np.round(np.exp(r.normal(11, 0.8, n)), 2)
        nv = o * np.where(r.random(n) < 1 / (1 + np.exp(-0.5 * sv)), r.uniform(1.0, 1.5, n), r.uniform(0.6, 1.0, n))
        old = [_j({'instalment': str(a)}) for a in o]
        new = [_j({'instalment': str(round(b, 2)), 'contract_code': k, 'creditor_type': c, 'outstanding_debt': str(round(b * 12, 2))})
               for b, k, c in zip(nv, cc, ct)]
        return old, new
    if code == 'dpd_chg':
        if dpd_old is None:
            dpd_old = np.floor(r.exponential(8 * np.exp(0.5 * sv))).astype(int)
            worse = r.random(n) < 1 / (1 + np.exp(-(0.2 + 0.9 * sv)))
            dpd_new = np.where(worse, dpd_old + r.integers(1, 31, n), 0)
        old = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(a)}) for k, c, a in zip(cc, ct, dpd_old)]
        new = [_j({'contract_code': k, 'creditor_type': c, 'overdue_instalment_count': int(b)}) for k, c, b in zip(cc, ct, dpd_new)]
        return old, new
    flags = {'st_gambler': 'is_gambler', 'st_recruit': 'is_recruit', 'st_stopcredit': 'stopcredit', 'st_bankrupt': 'is_bankrupt'}
    if code in flags:
        return [_j({flags[code]: 0})] * n, [_j({flags[code]: 1})] * n
    if code == 'settlement':
        return none, [_j({'is_settlement': 1})] * n
    if code == 'partial_ovd':
        return none, [_j({'has_half_overdue_bank_and_not_overdue_non_bank': True})] * n
    if code == 'refin_out_80':
        return none, [_j({'has_refinance_more_80_percent_ood': True})] * n
    if code == 'unsubscribe':
        return ([_j({'reason': None, 'status': 'Active', 'group_id': 66, 'removed_at': None})] * n,
                [_j({'reason': 'NO_ACTIVE_CONTRACTS', 'status': 'Removed', 'group_id': 66,
                     'removed_at': '2026-10-19T23:08:40.846181+05:00'})] * n)
    return none, none


def make_synthetic_data(n_clients=60_000, seed=SEED):
    """Synthetic ovrd_contr / hes / contracts with the production schema and a planted signal."""
    r = np.random.default_rng(seed)
    D = np.timedelta64(1, 'D')
    issue_from, issue_to = np.datetime64('2025-01-01'), np.datetime64('2026-10-15')
    data_end = np.datetime64('2026-11-18')
    hes_from, hes_to = np.datetime64('2026-09-07'), np.datetime64('2026-11-18')   # events created until 2026-11-17

    # clients: z = in-house risk, s = stress visible at other lenders (correlated with z)
    z = r.normal(size=n_clients)
    s = 0.5 * z + np.sqrt(0.75) * r.normal(size=n_clients)
    iin = np.array([f'{a:02d}{b:010d}' for a, b in zip(r.integers(60, 106, n_clients) % 100,
                                                       r.integers(0, 10**10, n_clients))])
    # contracts
    n_c = 1 + r.poisson(0.45, n_clients)
    ci = np.repeat(np.arange(n_clients), n_c)
    N = len(ci)
    open_d = issue_from + r.integers(0, int((issue_to - issue_from) / D) + 1, N) * D
    prod = r.choice(np.array(['16007', '16062', '16167', '16168']), N, p=[.35, .15, .25, .25])
    term_m = np.where(prod == '16167', r.choice([6, 12, 18], N),
             np.where(prod == '16168', r.choice([12, 24], N),
             np.where(prod == '16062', r.choice([12, 18, 24, 36], N), r.choice([24, 36, 48, 60], N))))
    end_d = open_d + np.round(term_m * 30.44).astype(int) * D
    mu = np.where(prod == '16167', 10.5, np.where(prod == '16168', 13.0, np.where(prod == '16062', 13.5, 14.5)))
    amount = np.round(np.exp(r.normal(mu, 0.6)), -2)
    early = r.random(N) < 0.2
    life = ((end_d - open_d) / D).astype(int)
    close_d = end_d.copy()
    close_d[early] = open_d[early] + np.maximum(30, (life[early] * r.uniform(0.15, 0.9, early.sum())).astype(int)) * D
    status = np.where(close_d <= data_end, 'Погашен', 'Актуален')
    pref = r.choice(np.array(['3199191', '1199221', '1340260', '3277770', '1677770', '2477770']), N)
    contract_number = np.array([f'{p}{i:09d}' for p, i in zip(pref, r.permutation(N) + 100_000)])
    pr = np.where(np.isin(prod, ['16007', '16062']), 'EKP', np.where(prod == '16167', 'CCC', 'CPA'))
    process_ref = np.array([f'{p}{i:09d}' if p == 'EKP' else f'{p}{i:012d}'
                            for p, i in zip(pr, r.permutation(N) + 10_000_000)])
    merchants = np.array([f'{x:012d}' for x in r.integers(10**11, 10**12, 300)])
    partner = np.where(prod == '16167', 'marketplace', np.where(prod == '16168', r.choice(merchants, N), None)).astype(object)
    prod_eff = pd.Series(prod).map({'16007': 0.0, '16062': 0.2, '16167': 0.35, '16168': 0.15}).values

    # delinquency episodes
    first_due = open_d + 30 * D
    active_days = np.maximum(0, ((np.minimum(close_d, data_end) - first_due) / D).astype(int))
    n_ep = r.poisson(0.0030 * np.exp(0.8 * z[ci] + prod_eff) * active_days)
    e_c = np.repeat(np.arange(N), n_ep)
    e_start = first_due[e_c] + (r.random(len(e_c)) * active_days[e_c]).astype(int) * D
    zc, sc = z[ci[e_c]], s[ci[e_c]]
    roll = r.random(len(e_c)) < 1 / (1 + np.exp(-(-2.4 + 1.0 * zc + 0.6 * sc + prod_eff[e_c])))
    e_dpd = np.where(roll, 31 + r.exponential(45, len(e_c)).astype(int),
                     1 + np.floor(29.999 * r.beta(0.7, np.clip(2.6 - 0.5 * sc, 1.2, 4.0))).astype(int))
    E = pd.DataFrame({'c': e_c, 'start': e_start, 'dpd': e_dpd, 'roll': roll}).sort_values(['c', 'start'])
    E['end'] = E['start'] + pd.to_timedelta(E['dpd'] - 1, unit='D')
    E['cm'] = E.groupby('c')['end'].cummax()
    prev = E.groupby('c')['cm'].shift()
    E = E[prev.isna() | (E['start'] > prev + pd.Timedelta(days=1))]
    E = E[E['start'] <= pd.Timestamp(data_end)].reset_index(drop=True)
    E['dpd_obs'] = np.minimum(E['dpd'], (pd.Timestamp(data_end) - E['start']).dt.days + 1)

    # DPD rows: last day of the episode, month ends and ~35% of other days (tests sparse layouts)
    L = E['dpd_obs'].values.astype(int)
    rep = np.repeat(np.arange(len(E)), L)
    off = np.arange(L.sum()) - np.repeat(np.cumsum(L) - L, L)
    act = E['start'].values.astype('datetime64[D]')[rep] + off * D
    is_last = off == (L[rep] - 1)
    is_month_end = (act + D).astype('datetime64[M]') != act.astype('datetime64[M]')
    keep = is_last | is_month_end | (r.random(len(off)) < 0.35)
    cidx = E['c'].values[rep][keep]
    act_k = act[keep]
    ns = lambda a: a.astype('datetime64[ns]')
    ovrd = pd.DataFrame({
        'rep_month': np.char.add(act_k.astype('datetime64[M]').astype(str), '-01').astype(object),
        'act_date': act_k.astype(str).astype(object),
        'contract_number': contract_number[cidx].astype(object),
        'ovd_days_cnt': (off[keep] + 1).astype(float),
        'process_ref': process_ref[cidx].astype(object),
        'iin': iin[ci[cidx]].astype(object),
        'open_date': ns(open_d[cidx]), 'close_date': ns(close_d[cidx]), 'end_date': ns(end_d[cidx]),
        'status_name': status[cidx].astype(object),
        'contract_amount': amount[cidx].astype(float),
        'partner_name': partner[cidx],
        'product_id': prod[cidx].astype(object)})
    dup = ovrd.sample(frac=0.01, random_state=seed).assign(status_name='Погашен')   # merge duplicates as in the real data
    ovrd = pd.concat([ovrd, dup], ignore_index=True).sort_values(['rep_month', 'act_date'], kind='mergesort').reset_index(drop=True)

    contracts = pd.DataFrame({
        'contract_number': contract_number.astype(object), 'process_ref': process_ref.astype(object),
        'iin': iin[ci].astype(object), 'open_date': ns(open_d), 'close_date': ns(close_d), 'end_date': ns(end_d),
        'status_name': status.astype(object), 'contract_amount': amount, 'partner_name': partner,
        'product_id': prod.astype(object)})

    # PKB monitoring events
    win_days = float((hes_to - hes_from) / D)
    act_client = np.zeros(n_clients, bool)
    act_client[ci[(open_d < hes_to) & (close_d > hes_from)]] = True
    rates = {   # other-lender events: (daily rate, effect of stress s)
        'app_online': (0.010, 0.7), 'lookup_other': (0.004, 0.5), 'report_req': (0.007, 0.6),
        'new_loan': (0.003, 0.5), 'new_contract': (0.003, 0.4), 'new_loan_refin': (0.0008, 0.6),
        'ovd_sum_chg': (0.004, 1.0), 'dpd_chg': (0.006, 1.1), 'debt_chg': (0.030, 0.1),
        'instalment_chg': (0.010, 0.2), 'collateral_chg': (0.001, 0.0), 'loan_closed': (0.005, -0.3),
        'settlement': (0.0008, 0.3), 'loan_assigned': (0.0003, 1.3), 'debt_terminated': (0.0002, 1.3),
        'prolong': (0.002, 0.8), 'restruct': (0.0005, 0.8), 'st_gambler': (0.0002, 0.9),
        'st_recruit': (0.0002, 0.0), 'st_stopcredit': (0.0002, 0.5), 'st_bankrupt': (0.00005, 1.5),
        'refin_out_80': (0.0004, -0.5), 'unsubscribe': (0.0003, 0.0)}
    parts = []

    def add(code, cl, t, codes=None, dpd_old=None, dpd_new=None):
        if len(cl) == 0:
            return
        old, new = _payloads(code, s[cl], r, codes, dpd_old, dpd_new)
        parts.append(pd.DataFrame({'cl': cl, 'code': code, 't': t, 'old': old, 'new': new}))

    t_unif = lambda n: pd.Timestamp(hes_from) + pd.to_timedelta(r.random(n) * win_days * 86400, unit='s')
    for code, (rate, beta) in rates.items():
        lam = rate * np.exp(beta * s - beta ** 2 / 2) * win_days * np.where(act_client, 1.0, 0.15)
        cl = np.repeat(np.arange(n_clients), r.poisson(lam))
        add(code, cl, t_unif(len(cl)))

    # events on our own contracts while they are delinquent, and stress bursts before an episode starts
    E['cl'] = ci[E['c'].values]
    hes_from_ts, hes_to_ts = pd.Timestamp(hes_from), pd.Timestamp(hes_to)
    Ew = E[(E['start'] > hes_from_ts - pd.Timedelta(days=60)) & (E['start'] < hes_to_ts)]
    own_cl, own_t, own_code, d_old, d_new = [], [], [], [], []
    for row in Ew.itertuples(index=False):
        k = contract_number[row.c] if r.random() < 0.6 else process_ref[row.c]
        for step in range(1, int(row.dpd_obs), 7):
            t = row.start + pd.Timedelta(days=step + 1, hours=float(r.uniform(0, 20)))
            if hes_from_ts <= t < hes_to_ts:
                own_cl.append(row.cl); own_t.append(t); own_code.append(k)
                d_old.append(max(step - 7, 0)); d_new.append(step)
    add('dpd_chg', np.array(own_cl, int), pd.to_datetime(own_t), own_code, np.array(d_old), np.array(d_new))
    m = (Ew['start'] + pd.Timedelta(days=2) >= hes_from_ts) & (Ew['start'] + pd.Timedelta(days=2) < hes_to_ts)
    po = Ew[m & (r.random(len(Ew)) < 0.35)]
    add('partial_ovd', po['cl'].values, po['start'] + pd.Timedelta(days=2, hours=9))
    Eb = E[(E['start'] >= hes_from_ts + pd.Timedelta(days=5)) & (E['start'] < hes_to_ts)]
    nb = r.poisson(np.where(Eb['roll'], 0.7, 0.35))
    bcl = np.repeat(Eb['cl'].values, nb)
    bst = np.repeat(Eb['start'].values, nb)
    bcode = r.choice(['app_online', 'report_req', 'dpd_chg', 'ovd_sum_chg', 'new_loan', 'lookup_other'],
                     len(bcl), p=[.3, .25, .15, .1, .1, .1])
    bt = pd.to_datetime(bst) - pd.to_timedelta(r.uniform(1, 21, len(bcl)) * 86400, unit='s')
    for code in np.unique(bcode):
        mm = bcode == code
        add(code, bcl[mm], bt[mm])

    ev = pd.concat(parts, ignore_index=True)
    ev = ev[(ev['t'] >= hes_from_ts) & (ev['t'] < hes_to_ts)].reset_index(drop=True)
    n = len(ev)
    created = pd.to_datetime(ev['t'].values)
    dv = (created.normalize() + pd.to_timedelta(r.integers(0, 2, n), unit='D'))
    hes = pd.DataFrame({
        'date_value': dv.strftime('%Y-%m-%d').astype(object),
        'date_change': dv + pd.Timedelta(days=1, hours=14, minutes=13, seconds=47),
        'internal_event_id': [str(uuid.UUID(bytes=r.bytes(16))) for _ in range(n)],
        'bin_iin': iin[ev['cl'].values].astype(object),
        'rule_name': ev['code'].map(SYN_RULE_NAMES).values.astype(object),
        'old_value': ev['old'].values, 'new_value': ev['new'].values,
        'event_date': created - pd.to_timedelta(r.uniform(0, 1.5, n) * 86400, unit='s'),
        'created_date': created})
    hes = hes.sort_values(['date_value', 'internal_event_id']).reset_index(drop=True)
    return ovrd, hes, contracts


if DEMO_MODE:
    with Timer('Generating synthetic data'):
        ovrd_contr, hes, contracts = make_synthetic_data()
else:
    # Your dataframes, taken from this kernel by name (nothing is copied).
    # If they live in another notebook, paste these cells there, or save and read them here, e.g.
    #   ovrd_contr = pd.read_pickle('ovrd_contr.pkl'); hes = pd.read_pickle('hes.pkl')
    # hes may also be an iterator of chunks, e.g. pd.read_sql(sql, conn, chunksize=2_000_000);
    # it is then filtered chunk by chunk in section 7 and never held in memory whole (set HES_START by hand).
    _g = globals()
    _missing = [v for v in (OVRD_VAR, HES_VAR) if v not in _g]
    if _missing:
        raise NameError(f'Dataframes {_missing} are not in this kernel. Load them first (or change OVRD_VAR / HES_VAR), '
                        'or set DEMO_MODE = True to run the synthetic demo.')
    ovrd_contr = _g[OVRD_VAR]
    hes = _g[HES_VAR]
    contracts = _g.get(CONTRACTS_VAR)
    if not isinstance(contracts, pd.DataFrame):
        contracts = None

print('ovrd_contr:', ovrd_contr.shape, '| contracts:', None if contracts is None else contracts.shape,
      '| hes:', hes.shape if isinstance(hes, pd.DataFrame) else 'iterator')
if DEMO_MODE:
    display(ovrd_contr.head(3))
    display(hes.head(3))

## 2b. Preflight validation of the input dataframes
Cheap checks on samples before any heavy step: required columns, parsable dates and DPD, IIN format, overlap of
clients between `hes` and `ovrd_contr`, JSON parse rate per rule, recognised rule names, and whether PKB contract
codes match our contract numbers. `FAIL` stops the notebook (`STOP_ON_PREFLIGHT_FAIL`), `WARN` only informs.

In [ ]:
def _sample_rows(df, n, seed=SEED):
    if len(df) <= n:
        return df
    pos = np.unique(np.random.default_rng(seed).integers(0, len(df), size=n))   # cheap on 85M rows
    return df.iloc[pos]


def preflight(ovrd, hes, contracts):
    rows = []

    def add(obj, check, status, detail):
        rows.append({'object': obj, 'check': check, 'status': status, 'detail': detail})

    def cols(df, obj, need, opt):
        miss = [c for c in need if c not in df.columns]
        add(obj, 'required columns', 'FAIL' if miss else 'OK', f'missing: {miss}' if miss else ', '.join(need))
        mo = [c for c in opt if c not in df.columns]
        if mo:
            add(obj, 'optional columns', 'WARN', f'missing: {mo} (features using them will be empty)')
        return not miss

    if not isinstance(ovrd, pd.DataFrame):
        add('ovrd_contr', 'type', 'FAIL', f'expected a DataFrame, got {type(ovrd).__name__}')
    elif cols(ovrd, 'ovrd_contr', [C_CONTRACT, C_IIN, C_ACT_DATE, C_DPD],
              [C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]):
        s = _sample_rows(ovrd, DQ_SAMPLE)
        ad = to_dt(s[C_ACT_DATE])
        r = ad.notna().mean()
        add('ovrd_contr', f'{C_ACT_DATE} parsable as date', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
            f'{r:.2%} parsed, range {ad.min()} .. {ad.max()}')
        d = pd.to_numeric(s[C_DPD], errors='coerce')
        add('ovrd_contr', f'{C_DPD} numeric and > 0', 'OK' if (d > 0).mean() > 0.95 else 'WARN',
            f'{(d > 0).mean():.2%} > 0, {d.isna().mean():.2%} missing, max {d.max():,.0f}')
        ii = norm_iin(s[C_IIN])
        r = ii.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
        add('ovrd_contr', f'{C_IIN} = 12 digits', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
        cn = norm_code(s[C_CONTRACT])
        add('ovrd_contr', f'{C_CONTRACT} filled', 'OK' if cn.notna().mean() > 0.99 else 'WARN', f'{cn.notna().mean():.2%}')
    if contracts is not None:
        cols(contracts, 'contracts', [C_CONTRACT, C_IIN, C_OPEN], [C_PROCESS_REF, C_CLOSE, C_END, C_STATUS, C_AMOUNT,
                                                                  C_PARTNER, C_PRODUCT])
    else:
        add('contracts', 'present', 'WARN', f'`{CONTRACTS_VAR}` not found: part B (early warning) is skipped')
    if isinstance(hes, pd.DataFrame):
        if cols(hes, 'hes', [H_IIN, H_RULE, H_NEW, EVENT_TIME_COL], [H_OLD, H_ID, H_EVENT_DATE]):
            s = _sample_rows(hes, min(DQ_SAMPLE, 300_000))
            t = s[EVENT_TIME_COL] if pd.api.types.is_datetime64_any_dtype(s[EVENT_TIME_COL]) else to_dt(s[EVENT_TIME_COL])
            r = t.notna().mean()
            add('hes', f'{EVENT_TIME_COL} parsable', 'OK' if r > 0.99 else 'WARN' if r > 0.9 else 'FAIL',
                f'{r:.2%} parsed, range {t.min()} .. {t.max()}')
            hi = norm_iin(s[H_IIN])
            r = hi.str.fullmatch(r'\d{12}').fillna(False).astype(bool).mean()
            add('hes', f'{H_IIN} = 12 digits', 'OK' if r > 0.99 else 'WARN' if r > 0.8 else 'FAIL', f'{r:.2%}')
            if isinstance(ovrd, pd.DataFrame) and C_IIN in ovrd.columns:
                known = set(norm_iin(pd.Series(pd.unique(ovrd[C_IIN].values))).dropna())
                r = hi.isin(known).mean()
                add('hes', 'clients also in ovrd_contr', 'OK' if r > 0.05 else 'FAIL',
                    f'{r:.2%} of sampled events belong to clients with overdue (0% means the IIN keys do not match)')
            rm = map_rule_names(pd.unique(s[H_RULE].dropna()))
            unk = [k for k, v in rm.items() if v.startswith('other_')]
            add('hes', 'rule names recognised', 'OK' if not unk else 'WARN',
                f'{len(rm) - len(unk)} of {len(rm)} recognised' + (f'; unknown (counted only): {unk[:5]}' if unk else ''))
            parsed = s[H_NEW].map(lambda v: _loads(v) is not None)
            by_rule = parsed.groupby(s[H_RULE].map(rm)).mean()
            bad = by_rule[by_rule < 0.5]
            add('hes', 'new_value JSON parsable', 'OK' if parsed.mean() > 0.9 and bad.empty else 'WARN',
                f'{parsed.mean():.2%} overall' + (f'; low for {bad.round(2).to_dict()}' if len(bad) else ''))
            if isinstance(ovrd, pd.DataFrame) and C_CONTRACT in ovrd.columns:
                dpd_rows = s[s[H_RULE].map(rm) == 'dpd_chg']
                codes = pd.Series([_parse_new(v)[0] for v in dpd_rows[H_NEW].values], dtype=object).dropna()
                own = set(norm_code(pd.Series(pd.unique(ovrd[C_CONTRACT].values))).dropna())
                if C_PROCESS_REF in ovrd.columns:
                    own |= set(norm_code(pd.Series(pd.unique(ovrd[C_PROCESS_REF].values))).dropna())
                r = codes.isin(own).mean() if len(codes) else np.nan
                add('hes', 'DPD-change events on our contracts', 'OK' if r == r and r > 0.01 else 'WARN',
                    f'{r:.2%} of sampled DPD-change events match our contract_number/process_ref '
                    '(near 0% means PKB uses another code; the own/other split would then be wrong)')
        add('hes', 'memory (shallow)', 'OK', f'{hes.memory_usage(index=False).sum() / 1e9:,.1f} GB without strings')
    else:
        add('hes', 'type', 'WARN', 'iterator of chunks: checks run on the fly; set HES_START by hand')
    res = pd.DataFrame(rows)
    display(res)
    return res


with Timer('Preflight checks'):
    preflight_result = preflight(ovrd_contr, hes, contracts)
if STOP_ON_PREFLIGHT_FAIL and (preflight_result['status'] == 'FAIL').any():
    raise ValueError('Preflight checks failed (rows with status FAIL above). Fix the column names in the first cell '
                     'or the data, or set STOP_ON_PREFLIGHT_FAIL = False to continue anyway.')

## 3. Contract attributes and the set of our own contract codes
PKB `contract_code` for our own loans appears either as `contract_number` (e.g. `3577770000000001`) or as
`process_ref` (e.g. `EKP000000001`), so both are collected. Duplicated contract rows (the same contract listed
once as Актуален and once as Погашен after the merge) are reduced to one row, preferring the closed status.

In [ ]:
def channel_of(partner):
    p = pd.Series(partner, copy=False).fillna('').astype(str).str.strip()
    out = np.where(p.isin(_NULLS), 'direct', np.where(p.str.fullmatch(r'\d{10,12}').fillna(False), 'merchant', p))
    return pd.Series(out, index=p.index)


def build_contract_info(ovrd, contracts=None):
    cols = [C_CONTRACT, C_IIN, C_PROCESS_REF, C_OPEN, C_CLOSE, C_END, C_STATUS, C_AMOUNT, C_PARTNER, C_PRODUCT]
    srcs = [x for x in (contracts, ovrd) if x is not None]
    frames = []
    for i, src in enumerate(srcs):
        c = [k for k in cols if k in src.columns]
        f = src[c].drop_duplicates()
        f = f.assign(_src=i)
        frames.append(f)
    ci = pd.concat(frames, ignore_index=True)
    ci['contract'] = norm_code(ci[C_CONTRACT]).values
    ci = ci.dropna(subset=['contract'])
    closed = ci[C_STATUS].astype(str).str.contains('Погаш|Закры|Спис', case=False, regex=True) if C_STATUS in ci else False
    ci['_closed'] = np.asarray(closed, dtype=int)
    ci = ci.sort_values(['contract', '_src', '_closed'], ascending=[True, True, False])
    ci = ci.groupby('contract', sort=False).first().reset_index()   # first non-null per column: contracts, then closed rows
    out = pd.DataFrame({'contract': ci['contract'].values})
    out['iin'] = norm_iin(ci[C_IIN]).values if C_IIN in ci else np.nan
    out['process_ref'] = norm_code(ci[C_PROCESS_REF]).values if C_PROCESS_REF in ci else np.nan
    for src, dst in ((C_OPEN, 'open_date'), (C_CLOSE, 'close_date'), (C_END, 'end_date')):
        out[dst] = to_dt(ci[src]).values if src in ci else pd.NaT
    out['amount'] = pd.to_numeric(ci[C_AMOUNT], errors='coerce').values if C_AMOUNT in ci else np.nan
    out['product'] = ci[C_PRODUCT].astype(str).values if C_PRODUCT in ci else 'NA'
    out['channel'] = channel_of(ci[C_PARTNER]).values if C_PARTNER in ci else 'NA'
    out['closed'] = ci['_closed'].values
    # True when the contract comes from the full `contracts` table; contracts seen only in ovrd_contr are known
    # to exist only from their first delinquency on (using them earlier would leak the future)
    out['from_contracts'] = (ci['_src'].values == 0) if contracts is not None else np.zeros(len(ci), dtype=bool)
    return out


with Timer('Contract attributes'):
    cinfo = build_contract_info(ovrd_contr, contracts)
    OWN_CODES = set(cinfo['contract'].dropna()) | set(cinfo['process_ref'].dropna())
print(f'{len(cinfo):,} contracts, {len(OWN_CODES):,} own contract codes')
display(cinfo.head(3))

## 4. Delinquency episodes and data-quality checks on `ovrd_contr`
For each row `dlq_start = act_date − (ovd_days_cnt − 1)`. Rows of the same contract whose `dlq_start` agree
(within `EPISODE_START_TOL_DAYS`) form one episode; its maximum DPD tells whether it rolled past 30.
The diagnostics show how dense the rows are (daily or sparse) and whether DPD grows one day per calendar day.

In [ ]:
def build_episodes(ovrd, tol_days=EPISODE_START_TOL_DAYS, data_end=None):
    d = pd.DataFrame({'contract': norm_code(ovrd[C_CONTRACT]).values,
                      'iin': norm_iin(ovrd[C_IIN]).values,
                      'act_date': to_dt(ovrd[C_ACT_DATE]).values,
                      'dpd': pd.to_numeric(ovrd[C_DPD], errors='coerce').values})
    n0 = len(d)
    d = d.dropna(subset=['contract', 'act_date', 'dpd'])
    d = d[d['dpd'] >= 1]
    if data_end is not None:
        d = d[d['act_date'] <= pd.Timestamp(data_end)]
    d['dpd'] = d['dpd'].round().astype('int32')
    # one row per contract-day: the merge with contract statuses can duplicate rows; keep the highest DPD
    d = (d.groupby(['contract', 'act_date'], sort=False)
           .agg(dpd=('dpd', 'max'), iin=('iin', 'first')).reset_index())
    d['dlq_start'] = d['act_date'] - pd.to_timedelta(d['dpd'] - 1, unit='D')
    d = d.sort_values(['contract', 'dlq_start', 'act_date'], kind='mergesort')
    gap = (d['dlq_start'] - d.groupby('contract')['dlq_start'].shift()).dt.days
    d['ep_seq'] = (gap.isna() | (gap > tol_days)).astype('int32').groupby(d['contract']).cumsum()
    ep = (d.groupby(['contract', 'ep_seq'], sort=False)
            .agg(iin=('iin', 'first'), dlq_start=('dlq_start', 'min'), max_dpd=('dpd', 'max'),
                 first_seen=('act_date', 'min'), last_seen=('act_date', 'max'), n_rows=('dpd', 'size'))
            .reset_index())
    ep = ep.sort_values(['contract', 'dlq_start'], kind='mergesort').reset_index(drop=True)
    # a DPD drop after a partial payment yields a 'new' dlq_start inside a still-running episode: not a new entry
    ep['overlap'] = (ep['dlq_start'] <= ep.groupby('contract')['last_seen'].shift()).values
    ep['ep_uid'] = np.arange(len(ep))
    stats = {'rows_in': n0, 'contract_days': len(d), 'episodes': len(ep),
             'contracts_with_overdue': ep['contract'].nunique(), 'clients_with_overdue': ep['iin'].nunique()}
    return ep, d, stats


with Timer('Delinquency episodes'):
    episodes, dpd_rows, ep_stats = build_episodes(ovrd_contr, data_end=DATA_END)
DATA_END_TS = pd.Timestamp(DATA_END) if DATA_END else dpd_rows['act_date'].max().normalize()
print({k: f'{v:,}' for k, v in ep_stats.items()}, '| DPD data end:', DATA_END_TS.date())

span = (episodes['last_seen'].clip(upper=DATA_END_TS) - episodes['dlq_start']).dt.days + 1
dq = pd.Series({
    'rows per episode-day (1.0 = daily rows)': (episodes['n_rows'] / span.clip(lower=1)).median(),
    'share of episodes seen in one row only': (episodes['n_rows'] == 1).mean(),
    'share of contract-days with jittered dlq_start': ((dpd_rows['dlq_start'] - dpd_rows.groupby(['contract', 'ep_seq'])['dlq_start'].transform('min')).dt.days > 0).mean(),
    'share of episodes reaching DPD 31+': (episodes['max_dpd'] >= 31).mean(),
    'share of episodes starting inside a running one (partial payment)': episodes['overlap'].mean(),
    'median max DPD': episodes['max_dpd'].median()})
display(dq.to_frame('value'))
del dpd_rows
gc.collect()

## 5. PKB monitoring coverage and data quality
Daily event volumes reveal when monitoring really started (the first days are often partial loads);
the delay between `event_date` (at PKB) and `created_date` (in the bank) shows how fresh each rule is.

In [ ]:
RULE_MAP = {}
if isinstance(hes, pd.DataFrame):
    with Timer('Daily and weekly volumes per rule (whole table, numpy counting)'):
        t_col = hes[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t_col):
            t_col = to_dt(t_col)
        day_num = t_col.values.astype('datetime64[D]').astype('int64')       # days since 1970-01-01, NaT -> min int
        ok = ~np.isnat(t_col.values)
        d0 = int(day_num[ok].min())
        daily_cnt = np.bincount(day_num[ok] - d0)
        daily = pd.Series(daily_cnt, index=pd.to_datetime(np.arange(len(daily_cnt)) + d0, unit='D'))
        daily = daily[daily > 0]
        if RUN_HES_DQ:
            rcode, runiq = pd.factorize(hes[H_RULE].values)                 # -1 = missing rule name
            RULE_MAP = map_rule_names(runiq)
            wk = (day_num - 4) // 7                                         # weeks starting on Monday
            w0 = int(wk[ok].min())
            nw = int(wk[ok].max()) - w0 + 1
            keep = ok & (rcode >= 0)
            cnt = np.bincount(rcode[keep] * nw + (wk[keep] - w0), minlength=len(runiq) * nw).reshape(len(runiq), nw)
            week_start = pd.to_datetime((np.arange(nw) + w0) * 7 + 4, unit='D')
            weekly_rule = (pd.DataFrame(cnt, index=[RULE_MAP[r] for r in runiq],
                                        columns=[d.strftime('%d.%m') for d in week_start])
                             .groupby(level=0).sum())
            del rcode, wk, keep
            if H_EVENT_DATE in hes.columns:
                smp = _sample_rows(hes[[H_RULE, H_EVENT_DATE, EVENT_TIME_COL]], DQ_SAMPLE)
                lag_h = (to_dt(smp[EVENT_TIME_COL]) - to_dt(smp[H_EVENT_DATE])) / np.timedelta64(1, 'h')
                lat = (pd.DataFrame({'rule': smp[H_RULE].map(RULE_MAP).values, 'lag_h': lag_h.values})
                         .groupby('rule')['lag_h'].describe(percentiles=[.5, .9, .99])[['count', '50%', '90%', '99%']])
                del smp
        del t_col, day_num, ok
        gc.collect()
    med = daily.median()
    HES_START_TS = pd.Timestamp(HES_START) if HES_START else daily[daily >= 0.5 * med].index.min()
    HES_END_TS = daily.index.max()
    print(f'events: {len(hes):,} | first day {daily.index.min().date()} | detected full coverage from {HES_START_TS.date()} '
          f'| last day {HES_END_TS.date()}')
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.plot(daily.index, daily.values, color=PAL['blue'])
    ax.axvline(HES_START_TS, color=PAL['gray'], lw=0.8)
    ax.set_title('PKB monitoring events per day (all rules)')
    ax.set_ylabel('events')
    plt.tight_layout(); plt.show()
    if RUN_HES_DQ:
        weekly_rule.insert(0, 'description', [RULE_DESC.get(c, '') for c in weekly_rule.index])
        print('Events per rule and week (week start, Monday):')
        display(weekly_rule)
        if H_EVENT_DATE in hes.columns:
            print(f'Delay {EVENT_TIME_COL} - {H_EVENT_DATE}, hours (sample of {DQ_SAMPLE:,} rows):')
            display(lat)
else:
    if HES_START is None:
        raise ValueError('hes is an iterator: set HES_START explicitly')
    HES_START_TS = pd.Timestamp(HES_START)
    HES_END_TS = DATA_END_TS
    print('hes is read in chunks: coverage and data-quality checks skipped')
print('Rule names found:', len(RULE_MAP), '| not recognised:', [k for k, v in RULE_MAP.items() if v.startswith('other_')])

## 6. Population A: entries into soft collection, target and in-house baseline features
* scoring day `score_date = dlq_start + OBS_DPD − 1`, from `HES_START + MIN_HISTORY_DAYS`;
* outcome must be observable: `dlq_start ≤ DATA_END − (BAD_DPD − 1)`;
* `target = 1` when the episode reaches DPD ≥ `BAD_DPD`.

**Baseline features** use only in-house data known on the scoring day: product, channel, amount, months on book,
share of term elapsed, the contract's and the client's previous delinquency (count, worst DPD as of the scoring day,
days since), other own contracts delinquent now, and the number of open own contracts.

In [ ]:
POP_START = HES_START_TS + days(MIN_HISTORY_DAYS)
LAST_START = DATA_END_TS - days(BAD_DPD - 1)


def build_collection_population(ep):
    p = ep[(ep['max_dpd'] >= OBS_DPD) & ~ep['overlap']].copy()
    p['score_date'] = p['dlq_start'] + days(OBS_DPD - 1)
    p = p[(p['score_date'] >= POP_START) & (p['dlq_start'] <= LAST_START) & (p['score_date'] <= HES_END_TS)]
    p['target'] = (p['max_dpd'] >= BAD_DPD).astype('int8')
    for name, k in EXTRA_TARGETS.items():
        observed = p['dlq_start'] <= DATA_END_TS - days(k - 1)
        p[name] = np.where(observed, (p['max_dpd'] >= k).astype(float), np.nan)
    p = p.dropna(subset=['iin']).reset_index(drop=True)
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


def build_baseline(pop, ep_all, cinfo):
    """In-house features as of the scoring day (no PKB data)."""
    b = pop[['ep_id', 'contract', 'iin', 'score_date']].merge(
        cinfo.drop(columns=['iin']), on='contract', how='left')
    out = pd.DataFrame({'ep_id': b['ep_id'].values})
    out['bl_product'] = b['product'].fillna('NA').astype(str).values
    out['bl_channel'] = b['channel'].fillna('NA').astype(str).values
    out['bl_amount'] = b['amount'].values
    mob = (b['score_date'] - b['open_date']).dt.days
    term = (b['end_date'] - b['open_date']).dt.days
    out['bl_mob_days'] = mob.values
    out['bl_term_days'] = term.values
    out['bl_term_elapsed'] = (mob / term.where(term > 0)).values
    hist = ep_all[['contract', 'iin', 'dlq_start', 'max_dpd', 'ep_uid']]
    for lvl, key in (('c', 'contract'), ('i', 'iin')):
        h = pop[['ep_id', key, 'score_date', 'ep_uid']].merge(
            hist[[key, 'dlq_start', 'max_dpd', 'ep_uid']], on=key, suffixes=('', '_h'))
        h = h[(h['dlq_start'] < h['score_date']) & (h['ep_uid_h'] != h['ep_uid'])]
        days_in = (h['score_date'] - h['dlq_start']).dt.days          # DPD the old episode would have yesterday
        h['dpd_asof'] = np.minimum(h['max_dpd'], days_in)
        h['end_asof'] = h['dlq_start'] + pd.to_timedelta(h['dpd_asof'] - 1, unit='D')
        h['ongoing'] = h['max_dpd'] >= days_in
        h['recent'] = h['dlq_start'] >= h['score_date'] - days(180)
        g = h.groupby('ep_id')
        agg = pd.DataFrame({
            f'bl_{lvl}_n_prev_ep': g.size(),
            f'bl_{lvl}_n_prev_ep_180d': g['recent'].sum(),
            f'bl_{lvl}_max_prev_dpd': g['dpd_asof'].max(),
            f'bl_{lvl}_days_since_prev': (g['score_date'].first() - g['end_asof'].max()).dt.days})
        if lvl == 'i':
            agg['bl_i_n_dlq_now'] = g['ongoing'].sum()
            agg['bl_i_n_prev_roll'] = (h['dpd_asof'] >= 31).groupby(h['ep_id']).sum()
        agg = agg.reindex(out['ep_id'].values)
        for c in agg.columns:
            vals = agg[c].values.astype(float)
            out[c] = vals if 'days_since' in c else np.nan_to_num(vals, nan=0.0)
    first_dlq = ep_all.groupby('contract')['dlq_start'].min().rename('first_dlq')
    oc = pop[['ep_id', 'iin', 'score_date']].merge(
        cinfo[['contract', 'iin', 'open_date', 'close_date', 'amount', 'from_contracts']].join(first_dlq, on='contract'),
        on='iin')
    known = oc['from_contracts'].astype(bool) | (oc['first_dlq'] <= oc['score_date'])
    oc = oc[known & (oc['open_date'] <= oc['score_date']) & ~(oc['close_date'] < oc['score_date'])]
    g = oc.groupby('ep_id')
    out['bl_i_n_open_contracts'] = g.size().reindex(out['ep_id'].values).fillna(0).values
    out['bl_i_open_amount'] = g['amount'].sum().reindex(out['ep_id'].values).fillna(0).values
    return out.set_index('ep_id')


with Timer('Population A and baseline features'):
    pop = build_collection_population(episodes)
    base_A = build_baseline(pop, episodes, cinfo)
print(f'Population A: {len(pop):,} episodes, {pop["iin"].nunique():,} clients, '
      f'scoring days {pop["score_date"].min().date()} .. {pop["score_date"].max().date()}, '
      f'bad rate {pop["target"].mean():.2%} ({int(pop["target"].sum()):,} bads)')
cohorts = pop.groupby('week').agg(episodes=('target', 'size'), bads=('target', 'sum'), bad_rate=('target', 'mean'),
                                  **{f'{k}_rate': (k, 'mean') for k in EXTRA_TARGETS})
display(cohorts)
if len(pop) == 0:
    raise ValueError('Empty population: check HES_START / MIN_HISTORY_DAYS / DATA_END')

### Population B (optional): current contracts, early warning
For each contract one random weekly snapshot inside the monitoring coverage is drawn. The contract must be open
and not delinquent on the snapshot day. `target = 1` if a delinquency episode starts within `EWS_HORIZON` days
and reaches DPD ≥ `EWS_MIN_DPD`. The open/closed status relies on `close_date`; if that column holds the planned
end date, contracts repaid early stay in the sample as "good", which dilutes but does not bias the comparison.

In [ ]:
def build_ews_population(cinfo, ep_all, rng):
    last_snap = DATA_END_TS - days(EWS_HORIZON + EWS_MIN_DPD - 2)
    snaps = pd.date_range(POP_START, min(last_snap, HES_END_TS), freq=f'{EWS_SNAPSHOT_FREQ_DAYS}D')
    if len(snaps) == 0:
        return None
    c = cinfo[cinfo['from_contracts'].astype(bool)].dropna(subset=['iin', 'open_date']).copy()   # complete issue list only
    c['score_date'] = snaps[rng.integers(0, len(snaps), len(c))]
    c = c[(c['open_date'] < c['score_date']) & ~(c['end_date'] < c['score_date'])
          & ~((c['closed'] == 1) & (c['close_date'] < c['score_date']))]
    if len(c) > EWS_MAX_ROWS:
        c = c.sample(n=EWS_MAX_ROWS, random_state=SEED)
    m = c[['contract', 'score_date']].merge(ep_all[['contract', 'dlq_start', 'max_dpd']], on='contract')
    dlq_now = (m['dlq_start'] < m['score_date']) & (m['dlq_start'] + pd.to_timedelta(m['max_dpd'] - 1, unit='D') >= m['score_date'] - days(1))
    future = ((m['dlq_start'] >= m['score_date']) & (m['dlq_start'] < m['score_date'] + days(EWS_HORIZON))
              & (m['max_dpd'] >= EWS_MIN_DPD))
    flags = pd.DataFrame({'contract': m['contract'], 'dlq_now': dlq_now, 'future': future}).groupby('contract').max()
    c = c.merge(flags, left_on='contract', right_index=True, how='left')
    c = c[~c['dlq_now'].fillna(False).astype(bool)]
    p = pd.DataFrame({'contract': c['contract'].values, 'iin': c['iin'].values, 'score_date': c['score_date'].values,
                      'target': c['future'].fillna(False).astype('int8').values})
    p['ep_uid'] = -1
    p['ep_id'] = np.arange(len(p))
    p['week'] = p['score_date'].dt.to_period('W-SUN').dt.start_time
    p['hist_days'] = (p['score_date'] - HES_START_TS).dt.days
    return p


pop_B, base_B = None, None
if RUN_EWS and contracts is not None:
    with Timer('Population B (early warning) and baseline features'):
        pop_B = build_ews_population(cinfo, episodes, np.random.default_rng(SEED))
        if pop_B is not None and len(pop_B):
            base_B = build_baseline(pop_B, episodes, cinfo)
    if pop_B is not None and len(pop_B):
        print(f'Population B: {len(pop_B):,} contract snapshots, bad rate {pop_B["target"].mean():.2%} '
              f'({int(pop_B["target"].sum()):,} entries into delinquency)')
else:
    print('Part B skipped (needs RUN_EWS = True and a `contracts` table)')

## 7. PKB events of the population clients: filter, parse JSON, mark own contracts
Only rows of population clients with `created_date` before the last scoring day are kept, which cuts the
full table down before any JSON is parsed. Parsed fields:

| field | JSON keys |
|---|---|
| `code` | `contract_code` (compared with our `contract_number` / `process_ref`) |
| `cred` | `creditor_type.id` (1 = bank, 2 = MFO) |
| `v_new`, `v_old` | first of `overdue_sum`, `debt_sum`, `instalment`, `overdue_instalment_count` |
| `f_new` | first `is_*` / `has_*` / `stopcredit` flag (statuses, settlement, refinancing) |
| `amt`, `debt`, `outst` | `summa_zaima`, `zadolzhenost`, `outstanding_debt` |
| `prolong_days` | `prolongation_end_date − prolongation_start_date` |

In [ ]:
def prefilter_hes(hes_src, iins, t_to):
    """Rows of `hes` for the given clients with event time < t_to. hes_src: DataFrame or iterable of DataFrames."""
    keys = set(pd.unique(pd.Series(list(iins))))
    keys_alt = keys | {k.lstrip('0') for k in keys}            # bin_iin stored without leading zeros
    chunks = [hes_src] if isinstance(hes_src, pd.DataFrame) else hes_src
    out = []
    for ch in chunks:
        t = ch[EVENT_TIME_COL]
        if not pd.api.types.is_datetime64_any_dtype(t):
            t = to_dt(t)
        idx = np.flatnonzero((t < t_to).values)
        is_num = pd.api.types.is_numeric_dtype(ch[H_IIN])
        raw = ch[H_IIN].values[idx]
        if is_num:
            m = pd.Series(raw).isin({int(k) for k in keys if k.isdigit()}).values
        else:
            probe = pd.Series(raw[:10_000]).astype(str)
            clean = probe.str.fullmatch(r'\d+').fillna(False).astype(bool).mean() > 0.999
            ser = pd.Series(raw) if clean else pd.Series(raw).astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            m = ser.isin(keys_alt).values
        idx = idx[m]
        cols = [H_IIN, H_RULE, H_OLD, H_NEW] + ([H_ID] if H_ID in ch.columns else [])
        part = ch.iloc[idx][cols].copy()
        part['t'] = t.values[idx]
        out.append(part)
    raw = pd.concat(out, ignore_index=True)
    if H_ID in raw.columns:
        n0 = len(raw)
        raw = raw.drop_duplicates(subset=[H_ID]).drop(columns=[H_ID])
        print(f'  duplicated {H_ID} removed: {n0 - len(raw):,}')
    raw['iin'] = norm_iin(raw[H_IIN]).values
    raw = raw[raw['iin'].isin(keys)].drop(columns=[H_IIN])
    return raw.reset_index(drop=True)


def parse_events(raw, own_codes, chunk=1_000_000):
    names = raw[H_RULE].astype('category')
    rmap = map_rule_names(names.cat.categories)
    RULE_MAP.update(rmap)
    rule = names.cat.rename_categories([rmap[c] for c in names.cat.categories]).astype(str).values
    parts = []
    for i0 in range(0, len(raw), chunk):
        ch = raw.iloc[i0:i0 + chunk]
        nw = list(zip(*[_parse_new(s) for s in ch[H_NEW].values]))
        od = list(zip(*[_parse_old(s) for s in ch[H_OLD].values]))
        parts.append(pd.DataFrame({
            'iin': ch['iin'].values, 't': ch['t'].values, 'rule': rule[i0:i0 + chunk],
            'code': np.array(nw[0], dtype=object), 'cred': np.array(nw[1], dtype='float32'),
            'v_new': np.array(nw[2], dtype=float), 'f_new': np.array(nw[3], dtype='float32'),
            'amt': np.array(nw[4], dtype=float), 'debt': np.array(nw[5], dtype=float),
            'outst': np.array(nw[6], dtype=float), 'prolong_days': np.array(nw[7], dtype='float32'),
            'v_old': np.array(od[0], dtype=float), 'f_old': np.array(od[1], dtype='float32')}))
        print(f'  parsed {min(i0 + chunk, len(raw)):,} / {len(raw):,}', flush=True)
    ev = pd.concat(parts, ignore_index=True)
    ev['own'] = np.where(ev['code'].isna(), -1, np.where(ev['code'].isin(own_codes), 1, 0)).astype('int8')
    return ev


iins_all = set(pop['iin'])
t_to = pop['score_date'].max()
if pop_B is not None:
    iins_all |= set(pop_B['iin'])
    t_to = max(t_to, pop_B['score_date'].max())
with Timer(f'Filtering hes to {len(iins_all):,} clients and events before {t_to.date()}'):
    hes_raw = prefilter_hes(hes, iins_all, t_to)
print(f'{len(hes_raw):,} rows kept')
with Timer('Parsing JSON'):
    ev = parse_events(hes_raw, OWN_CODES)
del hes_raw
gc.collect()

ev = ev[ev['rule'] != 'unsubscribe'].reset_index(drop=True)          # technical rule
ev['fam'] = ev['rule'].map(FAMILY).fillna('unknown')
code_share = ev.groupby('rule')['own'].apply(lambda x: (x >= 0).mean())
CONTRACT_RULES = set(code_share[code_share >= 0.5].index)               # rules that carry a contract_code
ev['rkey'] = np.where(ev['rule'].isin(CONTRACT_RULES),
                      ev['rule'] + np.where(ev['own'] == 1, '_own', '_oth'), ev['rule'])
ev['rkey'] = ev['rkey'].astype('category')
ev['oth'] = (ev['own'] != 1) & ~ev['rule'].isin(BANK_RULES)
ev = ev.drop(columns=['code'])
own_diag = ev.groupby('rule').agg(events=('own', 'size'), share_with_contract_code=('own', lambda x: (x >= 0).mean()),
                                  share_own_contract=('own', lambda x: (x == 1).mean()))
own_diag.insert(0, 'description', [RULE_DESC.get(c, '') for c in own_diag.index])
print('Own-contract matching per rule (share_own_contract near 0 for dpd_chg means the codes did not match):')
display(own_diag.sort_values('events', ascending=False))

## 8. PKB features
For every window W (7 / 14 / 30 days before the scoring day):
* `n_<rule>_<W>d`: number of events per rule; rules with a contract code are split into `_oth` (other lenders)
  and `_own` (our contracts);
* family counts (`fam_<family>_oth_n`), all events and distinct rules at other lenders;
* values from the JSON: worst DPD and number of DPD increases at other lenders, overdue amount (max, increases),
  principal (increases, decreases, net change), monthly payment change, application amounts, MFO share of
  applications, inquiries and new loans, prolongation days, status switches;
* `days_since_<rule>`: days since the latest event of the rule (whole history since monitoring start).

Feature groups: `pkb_other` (other lenders: the new information), `pkb_bank_rule` (rules defined relative to our
bank), `pkb_own` (our own contracts: repeats in-house data, used only as a leakage check), `baseline` (in-house).

In [ ]:
def rkey_group(rk):
    if rk.endswith('_own'):
        return 'pkb_own'
    base = rk[:-4] if rk.endswith('_oth') else rk
    return 'pkb_bank_rule' if base in BANK_RULES else 'pkb_other'


def rkey_desc(rk):
    base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
    tail = ' (other lenders)' if rk.endswith('_oth') else ' (our contracts)' if rk.endswith('_own') else ''
    return RULE_DESC.get(base, base) + tail


def build_pkb_features(pop, ev, windows=WINDOWS, lag_days=EVENT_LAG_DAYS):
    j = pop[['ep_id', 'iin', 'score_date']].merge(ev, on='iin', how='inner')
    cut = j['score_date'] - days(lag_days)
    j = j[(j['t'] < cut).values].copy()
    j['age'] = ((cut.loc[j.index] - j['t']) / np.timedelta64(1, 'D')).astype('float32')
    feats, meta = {}, {}

    def put(name, s, fill, group, family, rule, window, desc):
        feats[name] = (s, fill)
        meta[name] = dict(group=group, family=family, rule=rule, window=window, description=desc)

    for W in windows:
        sub = j[j['age'] <= W]
        cnt = sub.groupby(['ep_id', 'rkey'], observed=True).size().unstack(fill_value=0)
        for rk in cnt.columns:
            base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
            put(f'n_{rk}_{W}d', cnt[rk], 0, rkey_group(rk), FAMILY.get(base, 'unknown'), base, W,
                f'Number of events: {rkey_desc(rk)}, last {W} days')
        o = sub[sub['oth']]
        go = o.groupby('ep_id')
        put(f'all_oth_n_{W}d', go.size(), 0, 'pkb_other', 'all', 'all', W, f'All events at other lenders, last {W} days')
        put(f'all_oth_nrules_{W}d', go['rule'].nunique(), 0, 'pkb_other', 'all', 'all', W,
            f'Number of distinct rules triggered (other lenders), last {W} days')
        for fam in ('credit_hunger', 'delinquency', 'status', 'relief', 'positive', 'debt_burden'):
            x = o[o['fam'] == fam]
            put(f'fam_{fam}_oth_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', fam, 'family', W,
                f'Events of family "{fam}" (other lenders), last {W} days')

        def rows(rule, frame=o):
            return frame[frame['rule'] == rule]

        x = rows('dpd_chg')
        put(f'dpd_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Worst new DPD at other lenders, last {W} days')
        x_up = x[x['v_new'] > x['v_old'].fillna(0)]
        put(f'dpd_oth_n_up_{W}d', x_up.groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Number of DPD increases at other lenders, last {W} days')
        put(f'dpd_oth_n_ge30_{W}d', x[x['v_new'] >= 30].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'dpd_chg', W,
            f'Number of DPD events with DPD >= 30 at other lenders, last {W} days')
        x = rows('ovd_sum_chg')
        dl = x['v_new'] - x['v_old']
        put(f'ovdsum_oth_max_{W}d', x.groupby('ep_id')['v_new'].max(), NAN, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Largest overdue amount at other lenders, last {W} days')
        put(f'ovdsum_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Number of overdue-amount increases at other lenders, last {W} days')
        put(f'ovdsum_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'delinquency', 'ovd_sum_chg', W,
            f'Total growth of overdue amount at other lenders, last {W} days')
        x = rows('debt_chg')
        dl = x['v_new'] - x['v_old']
        put(f'debt_oth_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Number of principal increases at other lenders, last {W} days')
        put(f'debt_oth_n_down_{W}d', x[dl < 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Number of principal decreases at other lenders, last {W} days')
        put(f'debt_oth_up_sum_{W}d', dl.clip(lower=0).groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Total principal increase at other lenders, last {W} days')
        put(f'debt_oth_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'debt_chg', W,
            f'Net principal change at other lenders, last {W} days')
        x = rows('instalment_chg')
        dl = x['v_new'] - x['v_old']
        put(f'inst_net_{W}d', dl.groupby(x['ep_id']).sum(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Net change of total monthly payment, last {W} days')
        put(f'inst_n_up_{W}d', x[dl > 0].groupby('ep_id').size(), 0, 'pkb_other', 'debt_burden', 'instalment_chg', W,
            f'Number of monthly-payment increases, last {W} days')
        x = rows('app_online')
        put(f'app_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Total amount applied for online, last {W} days')
        put(f'app_amt_max_{W}d', x.groupby('ep_id')['amt'].max(), NAN, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Largest amount applied for online, last {W} days')
        put(f'app_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'app_online', W,
            f'Online applications to MFOs, last {W} days')
        x = rows('report_req')
        put(f'inq_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'report_req', W,
            f'Credit report requests by MFOs, last {W} days')
        x = o[o['rule'].isin(['new_loan', 'new_contract', 'new_loan_refin'])]
        put(f'newloan_n_mfo_{W}d', x[x['cred'] == 2].groupby('ep_id').size(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'New loans from MFOs, last {W} days')
        put(f'newloan_amt_sum_{W}d', x.groupby('ep_id')['amt'].sum(), 0, 'pkb_other', 'credit_hunger', 'new_loan', W,
            f'Total amount of new loans, last {W} days')
        x = rows('prolong')
        put(f'prolong_days_sum_{W}d', x.groupby('ep_id')['prolong_days'].sum(), 0, 'pkb_other', 'relief', 'prolong', W,
            f'Total prolongation days granted, last {W} days')
        x = o[o['rule'].isin(STATUS_RULES) & (o['f_new'] == 1)]
        put(f'status_on_n_{W}d', x.groupby('ep_id').size(), 0, 'pkb_other', 'status', 'status', W,
            f'Statuses switched on (gambler, conscript, stop-credit, bankrupt), last {W} days')
        own = sub[(sub['own'] == 1) & (sub['rule'] == 'dpd_chg')]
        put(f'dpd_own_max_{W}d', own.groupby('ep_id')['v_new'].max(), NAN, 'pkb_own', 'delinquency', 'dpd_chg', W,
            f'Worst DPD reported to PKB on our contracts, last {W} days')
    rec = j.groupby(['ep_id', 'rkey'], observed=True)['age'].min().unstack()
    for rk in rec.columns:
        base = rk[:-4] if rk.endswith(('_oth', '_own')) else rk
        put(f'days_since_{rk}', rec[rk], NAN, rkey_group(rk), FAMILY.get(base, 'unknown'), base, 'all',
            f'Days since the latest event: {rkey_desc(rk)}')
    put('days_since_any_oth', j[j['oth']].groupby('ep_id')['age'].min(), NAN, 'pkb_other', 'all', 'all', 'all',
        'Days since the latest event at other lenders')
    idx = pd.Index(pop['ep_id'].values, name='ep_id')
    cols = {}
    for name, (s, fill) in feats.items():
        v = s.reindex(idx)
        if fill == fill:
            v = v.fillna(fill)
        cols[name] = v.values.astype('float32')
    F = pd.DataFrame(cols, index=idx)
    keep = [c for c in F.columns if F[c].nunique(dropna=False) > 1]
    catalog = pd.DataFrame.from_dict(meta, orient='index').loc[keep]
    return F[keep], catalog


with Timer('PKB features, population A'):
    F_A, catalog = build_pkb_features(pop, ev)
print(f'{F_A.shape[1]} PKB features | share of episodes with any PKB event before scoring: '
      f'{F_A["days_since_any_oth"].notna().mean():.1%}')
display(catalog.groupby(['group', 'family']).size().unstack(fill_value=0))

## 9. Binning, WoE and IV
Numeric features: up to 20 quantile (or distinct-value) bins, small bins (< `MIN_BIN_SHARE`) merged into the
neighbour with the closest bad rate, adjacent bins merged until the bad rate is monotonic, then until at most
`MAX_BINS` remain. Missing values form their own bin. Categorical features: one bin per category, rare categories
pooled. `WoE = ln(%good / %bad)` with 0.5 smoothing, `IV = Σ (%good − %bad) · WoE`. Gini and KS are those of the
binned feature. A feature is flagged *unreliable* when fewer than `MIN_BADS_RELIABLE` bad cases sit outside its
largest bin (typical for rare events).

In [ ]:
def _fmt(v):
    if not np.isfinite(v):
        return 'inf'
    return f'{v:,.0f}' if abs(v - round(v)) < 1e-9 or abs(v) >= 100 else f'{v:,.3g}'


def fit_bins(x, y, max_bins=MAX_BINS, min_share=MIN_BIN_SHARE, monotonic=True, n_init=20):
    x = pd.Series(x, copy=False)
    y = np.asarray(y, dtype=float)
    N = len(y)
    if not pd.api.types.is_numeric_dtype(x) or pd.api.types.is_bool_dtype(x):
        s = x.astype(object)
        s = s.where(s.notna(), '__MISSING__').astype(str)
        vc = s.value_counts()
        cats = [c for c in vc.index if c != '__MISSING__' and vc[c] >= min_share * N]
        return {'kind': 'cat', 'cats': cats, 'n_codes': len(cats) + 2,
                'labels': cats + ['OTHER', 'MISSING']}
    xv = x.values.astype(float)
    miss = np.isnan(xv)
    v, yv = xv[~miss], y[~miss]
    if len(v) == 0:
        return {'kind': 'num', 'edges': np.array([]), 'n_codes': 2, 'labels': ['all', 'MISSING']}
    u = np.unique(v)
    if len(u) <= n_init:
        edges = u[:-1]
    else:
        edges = np.unique(np.quantile(v, np.linspace(0, 1, n_init + 1)[1:-1]))
        edges = edges[edges < u[-1]]
    k = len(edges) + 1
    code = np.searchsorted(edges, v, side='left')
    n = list(np.bincount(code, minlength=k).astype(float))
    b = list(np.bincount(code, weights=yv, minlength=k))
    hi = list(edges) + [np.inf]

    def merge(i):                      # merge bin i with bin i + 1
        n[i] += n[i + 1]; b[i] += b[i + 1]; hi[i] = hi[i + 1]
        del n[i + 1], b[i + 1], hi[i + 1]

    def br(i):
        return b[i] / n[i] if n[i] > 0 else 0.0

    min_n = max(min_share * N, 1)
    while len(n) > 1:
        i = int(np.argmin(n))
        if n[i] >= min_n:
            break
        if i == 0:
            merge(0)
        elif i == len(n) - 1:
            merge(i - 1)
        elif abs(br(i) - br(i - 1)) <= abs(br(i) - br(i + 1)):
            merge(i - 1)
        else:
            merge(i)
    if monotonic and len(n) > 2:
        rates = np.array([br(i) for i in range(len(n))])
        sign = np.sign(np.polyfit(np.arange(len(n)), rates, 1, w=np.sqrt(np.array(n)))[0]) or 1.0
        changed = True
        while changed and len(n) > 1:
            changed = False
            for i in range(len(n) - 1):
                if sign * (br(i + 1) - br(i)) < 0:
                    merge(i)
                    changed = True
                    break
    while len(n) > max_bins:
        merge(int(np.argmin([abs(br(i + 1) - br(i)) for i in range(len(n) - 1)])))
    edges = np.array(hi[:-1], dtype=float)
    labels = []
    lo = -np.inf
    for h in hi:
        labels.append(f'<= {_fmt(h)}' if lo == -np.inf and np.isfinite(h) else
                      f'> {_fmt(lo)}' if not np.isfinite(h) and lo != -np.inf else
                      'all' if not np.isfinite(h) else f'({_fmt(lo)}, {_fmt(h)}]')
        lo = h
    return {'kind': 'num', 'edges': edges, 'n_codes': len(edges) + 2, 'labels': labels + ['MISSING']}


def apply_bins(x, model):
    if model['kind'] == 'cat':
        s = pd.Series(x, copy=False).astype(object)
        lut = {c: i for i, c in enumerate(model['cats'])}
        k = len(model['cats'])
        codes = s.astype(str).map(lut).fillna(k).values.astype(int)
        codes[s.isna().values] = k + 1
        return codes
    xv = np.asarray(x, dtype=float)
    codes = np.searchsorted(model['edges'], xv, side='left')
    codes[np.isnan(xv)] = len(model['edges']) + 1
    return codes


def bin_stats(codes, y, n_codes):
    y = np.asarray(y, dtype=float)
    n = np.bincount(codes, minlength=n_codes).astype(float)
    b = np.bincount(codes, weights=y, minlength=n_codes)
    g = n - b
    ne = n > 0
    k = max(int(ne.sum()), 1)
    B, G = b.sum(), g.sum()
    db = (b + 0.5) / (B + 0.5 * k)
    dg = (g + 0.5) / (G + 0.5 * k)
    woe = np.where(ne, np.log(dg / db), 0.0)
    iv = np.where(ne, (dg - db) * woe, 0.0)
    return n, b, g, woe, iv


def binned_auc(n, b, risk):
    """AUC of a binned score; risk: one value per bin, higher = riskier (ties handled)."""
    g = n - b
    B, G = b.sum(), g.sum()
    if B == 0 or G == 0:
        return NAN
    o = np.argsort(risk, kind='mergesort')
    bs, gs, rs = b[o], g[o], np.asarray(risk)[o]
    auc, cum_g, i = 0.0, 0.0, 0
    while i < len(rs):
        j = i
        while j + 1 < len(rs) and rs[j + 1] == rs[i]:
            j += 1
        bb, gg = bs[i:j + 1].sum(), gs[i:j + 1].sum()
        auc += bb * (cum_g + 0.5 * gg)
        cum_g += gg
        i = j + 1
    return auc / (B * G)


def binned_ks(n, b, risk):
    g = n - b
    if b.sum() == 0 or g.sum() == 0:
        return NAN
    o = np.argsort(-np.asarray(risk), kind='mergesort')
    return float(np.max(np.abs(np.cumsum(b[o]) / b.sum() - np.cumsum(g[o]) / g.sum())))


def bin_table(x, y, model):
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    t = pd.DataFrame({'bin': model['labels'], 'n': n, 'bads': b, 'share': n / n.sum(),
                      'bad_rate': np.where(n > 0, b / np.maximum(n, 1), np.nan), 'woe': woe, 'iv': iv})
    return t[t['n'] > 0]


def feature_iv(x, y, model=None, **kw):
    model = model or fit_bins(x, y, **kw)
    codes = apply_bins(x, model)
    n, b, g, woe, iv = bin_stats(codes, y, model['n_codes'])
    ne = n > 0
    risk = -woe
    big = int(np.argmax(n))
    rate = np.where(ne, b / np.maximum(n, 1), np.nan)
    num_bins = [i for i in range(model['n_codes'] - 1) if ne[i]] if model['kind'] == 'num' else []
    direction = ''
    if len(num_bins) >= 2:
        direction = 'risk up with value' if rate[num_bins[-1]] > rate[num_bins[0]] else 'risk down with value'
    return model, dict(iv=float(iv.sum()), gini=2 * binned_auc(n[ne], b[ne], risk[ne]) - 1,
                       ks=binned_ks(n[ne], b[ne], risk[ne]), n_bins=int(ne.sum()),
                       bads_outside_main_bin=float(b.sum() - b[big]), direction=direction)


def compute_iv(X, y, features, catalog=None):
    y = np.asarray(y, dtype=float)
    rows, models = [], {}
    for f in features:
        x = X[f]
        m, r = feature_iv(x, y)
        models[f] = m
        xv = pd.to_numeric(x, errors='coerce') if m['kind'] == 'num' else None
        r.update(feature=f,
                 share_missing=float(pd.isna(x).mean()),
                 share_nonzero=float(((xv != 0) & xv.notna()).mean()) if xv is not None else NAN)
        rows.append(r)
    if not rows:
        return pd.DataFrame(columns=['iv', 'gini', 'ks', 'n_bins', 'bads_outside_main_bin', 'direction', 'share_missing',
                                     'share_nonzero', 'strength', 'reliable', 'group']), models
    t = pd.DataFrame(rows).set_index('feature')
    t['strength'] = t['iv'].map(iv_strength)
    t['reliable'] = t['bads_outside_main_bin'] >= MIN_BADS_RELIABLE
    if catalog is not None:
        t = t.join(catalog, how='left')
    t['group'] = t['group'].fillna('baseline') if 'group' in t else 'baseline'
    return t.sort_values('iv', ascending=False), models


def psi(expected, actual, eps=1e-4):
    e = np.asarray(expected, float) / max(np.sum(expected), 1)
    a = np.asarray(actual, float) / max(np.sum(actual), 1)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))

## 10. Results A1: which rules matter (event flags)
For every rule (split by other lenders / our contracts where a contract code exists) the flag "at least one event in
the last `IMPACT_WINDOW` days" is compared with the overall bad rate. **Lift** = bad rate of flagged episodes /
overall bad rate. This is the table to discuss with the collection business: how often the event happens and how
much it moves the roll rate.

In [ ]:
y_A = pop.set_index('ep_id').loc[F_A.index, 'target'].values.astype(float)
BR_A = y_A.mean()


def rule_impact(F, y, catalog, window=IMPACT_WINDOW):
    y = np.asarray(y, float)
    rows = []
    br = y.mean()
    for f, meta in catalog.iterrows():
        if not (f.startswith('n_') and meta['window'] == window):
            continue
        flag = (F[f].values > 0)
        nf = int(flag.sum())
        if nf == 0:
            continue
        bf = float(y[flag].sum())
        n_, b_, g_, woe, iv = bin_stats(flag.astype(int), y, 2)
        rows.append(dict(rule_key=f[2:-(len(str(window)) + 2)], group=meta['group'], family=meta['family'],
                         description=rkey_desc(f[2:-(len(str(window)) + 2)]),
                         share_flagged=nf / len(y), n_flagged=nf, bads_flagged=int(bf),
                         bad_rate_flagged=bf / nf, bad_rate_not=float(y[~flag].mean()) if nf < len(y) else NAN,
                         lift=(bf / nf) / br if br > 0 else NAN, iv_flag=float(iv.sum()),
                         reliable=bf >= MIN_BADS_RELIABLE / 2 and (len(y) - nf) > 0))
    if not rows:
        return pd.DataFrame(columns=['rule_key', 'group', 'family', 'description', 'share_flagged', 'n_flagged',
                                     'bads_flagged', 'bad_rate_flagged', 'bad_rate_not', 'lift', 'iv_flag', 'reliable'])
    return pd.DataFrame(rows).sort_values('iv_flag', ascending=False).reset_index(drop=True)


impact_A = rule_impact(F_A, y_A, catalog)
display(impact_A)


def plot_rule_lift(impact, title, min_share=0.001):
    d = impact[impact['share_flagged'] >= min_share].sort_values('lift')
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(d) + 1.2))
    col = np.where(d['lift'] >= 1, PAL['red'], PAL['blue'])
    ax.barh(d['rule_key'], d['lift'] - 1, left=1, color=col, height=0.6)
    ax.axvline(1, color=PAL['muted'], lw=0.8)
    for i, (l, s, rel) in enumerate(zip(d['lift'], d['share_flagged'], d['reliable'])):
        ax.text(max(l, 1) + 0.03, i, f'{s:.1%} flagged' + ('' if rel else ', few bads'), va='center',
                fontsize=7, color=PAL['muted'])
    ax.set_xlabel('lift = bad rate when the event happened / overall bad rate')
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(); plt.show()


plot_rule_lift(impact_A, f'Population A: roll to DPD {BAD_DPD}+, event in last {IMPACT_WINDOW} days '
                         '(red = riskier, blue = safer)')

## 11. Results A2: IV of every feature
All PKB features and the in-house baseline features on the full population A.

In [ ]:
X_A = F_A.join(base_A)
features_A = list(X_A.columns)
with Timer(f'IV for {len(features_A)} features'):
    iv_A, models_A = compute_iv(X_A, y_A, features_A, catalog)
cols_show = ['group', 'family', 'window', 'iv', 'strength', 'gini', 'ks', 'n_bins', 'share_nonzero', 'share_missing',
             'bads_outside_main_bin', 'reliable', 'direction', 'description']
print('Top 40 features by IV:')
display(iv_A[cols_show].head(40))
print('IV by group (number of features per strength class):')
display(pd.crosstab(iv_A['group'], iv_A['strength']))


def plot_iv(iv, title, top=30):
    d = iv.head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.27 * len(d) + 1.2))
    ax.barh(d.index, d['iv'], color=[GROUP_COLOR.get(g, PAL['gray']) for g in d['group']], height=0.62)
    for thr in (0.02, 0.1, 0.3):
        ax.axvline(thr, color='#c9c8c2', lw=0.8)
    groups = [g for g in GROUP_COLOR if g in set(d['group'])]
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOR[g]) for g in groups], labels=groups,
              loc='lower right')
    ax.set_xlabel('Information Value (lines at 0.02 / 0.1 / 0.3)')
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout(); plt.show()


plot_iv(iv_A, 'Population A: top features by IV')
pkb_groups = ('pkb_other', 'pkb_bank_rule')
plot_iv(iv_A[iv_A['group'].isin(pkb_groups)], 'Population A: top PKB features (other lenders and bank rules)')

### WoE profiles of the strongest PKB features
Bars: bad rate per bin (bin share above each bar). Horizontal line: overall bad rate.

In [ ]:
def plot_woe_grid(X, y, models, feats, ncols=3, title=''):
    feats = list(feats)
    if not feats:
        return
    nrows = int(math.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 2.9 * nrows), squeeze=False)
    br = np.mean(y)
    for ax, f in zip(axes.ravel(), feats):
        t = bin_table(X[f], y, models[f])
        ax.bar(range(len(t)), t['bad_rate'], color=PAL['blue'], width=0.6)
        ax.axhline(br, color=PAL['muted'], lw=0.8)
        for i, s in enumerate(t['share']):
            ax.text(i, t['bad_rate'].iloc[i], f'{s:.0%}', ha='center', va='bottom', fontsize=7, color=PAL['muted'])
        ax.set_xticks(range(len(t)))
        ax.set_xticklabels(t['bin'], rotation=30, ha='right', fontsize=7)
        ax.set_title(f'{f}\nIV {bin_table(X[f], y, models[f])["iv"].sum():.3f}', fontsize=8.5)
        ax.grid(axis='x', visible=False)
    for ax in axes.ravel()[len(feats):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=10.5, fontweight='bold')
    plt.tight_layout(); plt.show()


top_pkb = iv_A[iv_A['group'].isin(pkb_groups) & iv_A['reliable']].head(12).index
plot_woe_grid(X_A, y_A, models_A, top_pkb, title='Bad rate by bin: strongest PKB features')
print('Bin tables of the top 5 PKB features:')
for f in top_pkb[:5]:
    print(f'\n{f}: {catalog.loc[f, "description"]}')
    display(bin_table(X_A[f], y_A, models_A[f]))

## 12. Stability
* **Weekly IV and Gini** with the bins fixed on the full sample; Gini per week uses the full-sample WoE order,
  so a sign flip shows up as a low or negative value.
* **PSI** of each week's bin distribution against the first week.
* **Full 30-day history**: 30-day features of episodes scored in the first weeks of monitoring see less than 30
  days of events; their IV is recomputed on episodes with complete history.
* **IV by product**.

In [ ]:
def weekly_stability(X, y, week, models, feats):
    y = np.asarray(y, float)
    wk = np.sort(pd.unique(week))
    rows = []
    for f in feats:
        m = models[f]
        codes = apply_bins(X[f], m)
        n_all, b_all, _, woe_all, _ = bin_stats(codes, y, m['n_codes'])
        ref = None
        ivs, ginis, psis = [], [], []
        for w in wk:
            mk = (week == w)
            n, b, g, woe, iv = bin_stats(codes[mk], y[mk], m['n_codes'])
            ne = n > 0
            ivs.append(iv.sum())
            ginis.append(2 * binned_auc(n[ne], b[ne], -woe_all[ne]) - 1)
            if ref is None:
                ref = n
            psis.append(psi(ref, n))
        rows.append(dict(feature=f, iv_min=np.min(ivs), iv_max=np.max(ivs), iv_cv=np.std(ivs) / max(np.mean(ivs), 1e-9),
                         gini_min_week=np.nanmin(ginis), gini_max_week=np.nanmax(ginis), psi_max=np.max(psis),
                         **{f'iv_{pd.Timestamp(w):%d.%m}': v for w, v in zip(wk, ivs)}))
    if not rows:
        return pd.DataFrame(columns=['stable'])
    t = pd.DataFrame(rows).set_index('feature')
    t['stable'] = (t['gini_min_week'] > 0) & (t['psi_max'] < 0.25)
    return t


stab_feats = list(iv_A[(iv_A['iv'] >= IV_SELECT) & iv_A['reliable']].index[:40])
week_A = pop.set_index('ep_id').loc[X_A.index, 'week'].values
stab_A = weekly_stability(X_A, y_A, week_A, models_A, stab_feats)
display(iv_A.loc[stab_feats, ['group', 'iv']].join(stab_A))

top6 = [f for f in iv_A[iv_A['group'].isin(pkb_groups)].index if f in stab_A.index][:6]
if top6:
    wcols = [c for c in stab_A.columns if c.startswith('iv_') and c[3:5].isdigit()]
    fig, ax = plt.subplots(figsize=(9, 3))
    for i, f in enumerate(top6):
        ax.plot([c[3:] for c in wcols], stab_A.loc[f, wcols].values, color=SERIES[i], marker='o', ms=4, label=f)
    ax.set_title('Weekly IV of the top PKB features (bins fixed on the full sample)')
    ax.set_xlabel('scoring week (start)')
    ax.set_ylabel('IV')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1))
    plt.tight_layout(); plt.show()

full_hist = pop.set_index('ep_id').loc[X_A.index, 'hist_days'].values >= max(WINDOWS)
f30 = [f for f in iv_A.index if catalog.reindex([f])['window'].iloc[0] == max(WINDOWS)]
if full_hist.sum() > 0 and full_hist.mean() < 1 and f30:
    y_fh = y_A[full_hist]
    iv_fh = compute_iv(X_A.loc[full_hist], y_fh, f30)[0]['iv'] if y_fh.sum() > 0 else pd.Series(dtype=float)
    comp = pd.DataFrame({'iv_all_episodes': iv_A.loc[f30, 'iv'], 'iv_full_30d_history': iv_fh}).sort_values('iv_all_episodes', ascending=False)
    print(f'{max(WINDOWS)}-day features: {full_hist.sum():,} episodes ({full_hist.mean():.0%}) have the complete look-back')
    display(comp.head(20))

prod_A = base_A.loc[X_A.index, 'bl_product'].values
top_iv_feats = list(iv_A[iv_A['group'].isin(pkb_groups)].head(15).index)
by_prod = {}
for p_ in pd.Series(prod_A).value_counts().index[:6]:
    mk = prod_A == p_
    if y_A[mk].sum() < 20:
        continue
    by_prod[f'{p_} (n={mk.sum():,}, bad {y_A[mk].mean():.1%})'] = {
        f: bin_stats(apply_bins(X_A.loc[mk, f], models_A[f]), y_A[mk], models_A[f]['n_codes'])[4].sum() for f in top_iv_feats}
print('IV of the top PKB features by product (full-sample bins):')
display(pd.DataFrame(by_prod))

## 13. Short-list: correlation filter
Candidates: PKB features of groups `pkb_other` and `pkb_bank_rule` with IV ≥ `IV_SELECT`, reliable, stable.
Greedy selection by IV, skipping features whose Spearman correlation with an already selected one exceeds `CORR_MAX`.

In [ ]:
def spearman(X, feats):
    """Spearman correlation; a missing value means "no event": +inf for days_since_*, -1 otherwise."""
    D = X[feats].astype(float)
    for f in feats:
        D[f] = D[f].fillna(1e9 if f.startswith('days_since') else -1)
    return D.rank().corr()


def greedy_select(iv_series, corr, corr_max=CORR_MAX, max_n=MAX_MODEL_FEATURES):
    sel = []
    for f in iv_series.index:
        if all(abs(corr.loc[f, s]) < corr_max for s in sel):
            sel.append(f)
        if len(sel) >= max_n:
            break
    return sel


cand = iv_A[iv_A['group'].isin(pkb_groups) & (iv_A['iv'] >= IV_SELECT) & iv_A['reliable']]
cand = cand[[stab_A['stable'].get(f, True) for f in cand.index]]
if len(cand):
    corr_A = spearman(X_A, list(cand.index[:60]))
    shortlist = greedy_select(cand['iv'].iloc[:60], corr_A)
    print(f'{len(cand)} candidates -> short-list of {len(shortlist)}:')
    display(iv_A.loc[shortlist, ['group', 'family', 'window', 'iv', 'gini', 'share_nonzero', 'description']])
    sl = shortlist[:15]
    fig, ax = plt.subplots(figsize=(0.45 * len(sl) + 3, 0.45 * len(sl) + 2))
    im = ax.imshow(corr_A.loc[sl, sl].values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(sl))); ax.set_xticklabels(sl, rotation=60, ha='right', fontsize=7)
    ax.set_yticks(range(len(sl))); ax.set_yticklabels(sl, fontsize=7)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label='Spearman')
    ax.set_title('Correlation of the short-listed PKB features')
    plt.tight_layout(); plt.show()
else:
    shortlist = []
    print('No PKB feature passes the IV / reliability / stability filters')

## 14. Incremental value over the in-house baseline (out-of-time)
* split by scoring day: first `TRAIN_SHARE` of days = train, the rest = test (out of time);
* bins, IV and feature selection are refitted **on train only**;
* **Model A**: logistic regression on WoE of the baseline features;
  **Model B** = A + selected PKB features (`pkb_other`, `pkb_bank_rule`);
  **Model C** = B + PKB events on our own contracts (shows what the raw feed adds, mostly duplicated in-house data);
* ΔGini on test with a bootstrap 95% interval (resampling clients, so several contracts of one client stay together);
* cross-check with gradient boosting on all features (no selection).

In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    try:
        from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
        from sklearn.ensemble import HistGradientBoostingClassifier
    except ImportError:
        HistGradientBoostingClassifier = None


def gini(y, s, w=None):
    return 2 * roc_auc_score(y, s, sample_weight=w) - 1


def capture(y, s, q):
    o = np.argsort(-np.asarray(s), kind='mergesort')
    k = max(int(np.ceil(q * len(y))), 1)
    return float(np.asarray(y)[o][:k].sum() / max(np.sum(y), 1))


def boot_delta(y, s_a, s_b, clusters, n_boot=N_BOOT, seed=SEED):
    codes, uniq = pd.factorize(pd.Series(clusters))
    nc = len(uniq)
    r = np.random.default_rng(seed)
    out = []
    for _ in range(n_boot):
        w = np.bincount(r.integers(0, nc, nc), minlength=nc)[codes].astype(float)
        mk = w > 0
        if len(np.unique(y[mk])) < 2:
            continue
        out.append(gini(y[mk], s_b[mk], w[mk]) - gini(y[mk], s_a[mk], w[mk]))
    return np.percentile(out, [2.5, 50, 97.5]) if out else np.array([NAN] * 3)


def select_on_train(Xtr, ytr, feats, max_n=MAX_MODEL_FEATURES, iv_min=IV_SELECT):
    if not feats:
        return [], pd.DataFrame()
    t, _ = compute_iv(Xtr, ytr, feats)
    t = t[(t['iv'] >= iv_min) & t['reliable']]
    if t.empty:
        return [], t
    num = [f for f in t.index[:60] if pd.api.types.is_numeric_dtype(Xtr[f])]
    corr = spearman(Xtr, num) if num else None
    sel = []
    for f in t.index[:60]:
        if corr is not None and f in num and any(s in num and abs(corr.loc[f, s]) >= CORR_MAX for s in sel):
            continue
        sel.append(f)
        if len(sel) >= max_n:
            break
    return sel, t


def woe_fit(Xtr, ytr, feats):
    enc = {}
    for f in feats:
        m = fit_bins(Xtr[f], ytr)
        enc[f] = (m, bin_stats(apply_bins(Xtr[f], m), ytr, m['n_codes'])[3])
    return enc


def woe_apply(X, enc, feats):
    return np.column_stack([enc[f][1][apply_bins(X[f], enc[f][0])] for f in feats]) if feats else np.zeros((len(X), 0))


def incremental_value(X, y, score_date, clusters, base_feats, pkb_feats, own_feats, label):
    y = np.asarray(y, float)
    cut = pd.Series(score_date).quantile(TRAIN_SHARE)
    tr = (pd.Series(score_date) <= cut).values
    te = ~tr
    if y[te].sum() < 10 or y[tr].sum() < 10:
        print('Not enough bads for an out-of-time test'); return None
    Xtr, Xte, ytr, yte = X.loc[tr], X.loc[te], y[tr], y[te]
    print(f'{label}: train {tr.sum():,} (bad {ytr.mean():.2%}, up to {pd.Timestamp(cut).date()}), '
          f'test {te.sum():,} (bad {yte.mean():.2%})')
    sel_base, _ = select_on_train(Xtr, ytr, base_feats, max_n=30, iv_min=0.005)
    sel_pkb, iv_tr = select_on_train(Xtr, ytr, pkb_feats)
    sel_own, _ = select_on_train(Xtr, ytr, own_feats, max_n=5)
    sets = {'A: baseline': sel_base, 'B: + PKB other lenders': sel_base + sel_pkb,
            'C: + PKB own contracts': sel_base + sel_pkb + sel_own}
    enc = woe_fit(Xtr, ytr, sorted(set(sets['C: + PKB own contracts'])))
    res, scores, coefs = [], {}, None
    for name, fs in sets.items():
        if not fs:
            continue
        lr = LogisticRegression(C=1.0, max_iter=2000)
        lr.fit(woe_apply(Xtr, enc, fs), ytr)
        p_tr = lr.predict_proba(woe_apply(Xtr, enc, fs))[:, 1]
        p_te = lr.predict_proba(woe_apply(Xte, enc, fs))[:, 1]
        scores[name] = p_te
        res.append({'model': name, 'n_features': len(fs), 'gini_train': gini(ytr, p_tr), 'gini_test': gini(yte, p_te),
                    'capture_top10%': capture(yte, p_te, 0.1), 'capture_top20%': capture(yte, p_te, 0.2),
                    'capture_top30%': capture(yte, p_te, 0.3)})
        if name.startswith('B'):
            coefs = pd.DataFrame({'feature': fs, 'coef': lr.coef_[0]}).set_index('feature')
            coefs['group'] = ['baseline' if f.startswith('bl_') else catalog.loc[f, 'group'] for f in fs]
            coefs['sign_ok'] = coefs['coef'] < 0          # WoE > 0 means lower risk, so coefficients should be negative
    if not res:
        print(f'{label}: no features selected on train')
        return None
    res = pd.DataFrame(res).set_index('model')
    cl_te = np.asarray(clusters)[te]
    for name in (res.index[1:] if 'A: baseline' in scores else []):
        lo, mid, hi = boot_delta(yte, scores['A: baseline'], scores[name], cl_te)
        res.loc[name, 'delta_gini_vs_A'] = res.loc[name, 'gini_test'] - res.loc['A: baseline', 'gini_test']
        res.loc[name, 'delta_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    if RUN_HGB and HistGradientBoostingClassifier is not None:
        def hgb_matrix(Xs, fs):
            cat = [f for f in fs if not pd.api.types.is_numeric_dtype(X[f])]
            num = [f for f in fs if f not in cat]
            return np.column_stack([Xs[num].astype(float).values, woe_apply(Xs, enc_cat, cat)]) if cat else Xs[num].astype(float).values
        cat_feats = [f for f in base_feats if not pd.api.types.is_numeric_dtype(X[f])]
        enc_cat = woe_fit(Xtr, ytr, cat_feats)
        hg = {}
        for name, fs in (('A: baseline', base_feats), ('B: + PKB other lenders', base_feats + pkb_feats)):
            m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                               min_samples_leaf=max(50, int(0.002 * tr.sum())),
                                               l2_regularization=1.0, random_state=SEED)
            m.fit(hgb_matrix(Xtr, fs), ytr)
            hg[name] = m.predict_proba(hgb_matrix(Xte, fs))[:, 1]
            res.loc[name, 'gini_test_boosting'] = gini(yte, hg[name])
        lo, mid, hi = boot_delta(yte, hg['A: baseline'], hg['B: + PKB other lenders'], cl_te) if pkb_feats else (NAN,) * 3
        res.loc['B: + PKB other lenders', 'delta_boosting_CI95'] = f'[{lo:+.4f}, {hi:+.4f}]'
    display(res)
    if coefs is not None:
        print('Model B coefficients (WoE scale; negative is the expected sign):')
        display(coefs.join(iv_tr[['iv']], how='left') if 'iv' in iv_tr.columns else coefs)
    if len(scores) >= 2:
        fig, ax = plt.subplots(figsize=(5.2, 4))
        for (name, s), col in zip(list(scores.items())[:2], (PAL['gray'], PAL['blue'])):
            o = np.argsort(-s, kind='mergesort')
            cum = np.cumsum(yte[o]) / yte.sum()
            ax.plot(np.arange(1, len(o) + 1) / len(o), cum, color=col, label=f'{name} (Gini {gini(yte, s):.3f})')
        ax.plot([0, 1], [0, 1], color='#c9c8c2', lw=0.8)
        ax.set_xlabel('share of episodes, riskiest first')
        ax.set_ylabel('share of bads captured')
        ax.set_title(f'{label}: out-of-time gains')
        ax.legend(loc='lower right')
        plt.tight_layout(); plt.show()
    return res, sets


base_feats_A = list(base_A.columns)
pkb_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] in pkb_groups]
own_feats_A = [f for f in F_A.columns if catalog.loc[f, 'group'] == 'pkb_own']
sd_A = pop.set_index('ep_id').loc[X_A.index, 'score_date'].values
iin_A = pop.set_index('ep_id').loc[X_A.index, 'iin'].values
with Timer('Incremental value, population A'):
    inc_A = incremental_value(X_A, y_A, sd_A, iin_A, base_feats_A, pkb_feats_A, own_feats_A, 'Population A')

## 15. Secondary targets
IV of the same features against the other outcome definitions (bins refitted per target).

In [ ]:
sec = {}
for name in EXTRA_TARGETS:
    yy = pop.set_index('ep_id').loc[X_A.index, name].values
    mk = ~np.isnan(yy)
    if mk.sum() == 0 or np.nansum(yy) < 20:
        print(f'{name}: not enough observed outcomes yet')
        continue
    t, _ = compute_iv(X_A.loc[mk], yy[mk], features_A)
    sec[f'iv_{name}'] = t['iv']
    print(f'{name}: {mk.sum():,} episodes, bad rate {np.nanmean(yy[mk]):.2%}')
if sec:
    iv_targets = iv_A[['group', 'iv']].rename(columns={'iv': f'iv_roll{BAD_DPD}'}).join(pd.DataFrame(sec))
    display(iv_targets[iv_targets['group'].isin(pkb_groups)].head(25))

## 16. Part B: early warning for current contracts
The same PKB features for contracts that are not delinquent on the snapshot day. Target: entry into delinquency
within `EWS_HORIZON` days. Shows whether PKB events can trigger a pre-delinquency contact.

In [ ]:
iv_B, impact_B, inc_B = None, None, None
if pop_B is not None and len(pop_B) and pop_B['target'].sum() >= 20:
    with Timer('PKB features, population B'):
        F_B, catalog_B = build_pkb_features(pop_B, ev)
    X_B = F_B.join(base_B)
    y_B = pop_B.set_index('ep_id').loc[X_B.index, 'target'].values.astype(float)
    print(f'Population B: {len(y_B):,} snapshots, bad rate {y_B.mean():.2%}')
    impact_B = rule_impact(F_B, y_B, catalog_B)
    display(impact_B.head(25))
    plot_rule_lift(impact_B, f'Population B: entry into delinquency within {EWS_HORIZON} days, event in last {IMPACT_WINDOW} days')
    iv_B, models_B = compute_iv(X_B, y_B, list(X_B.columns), catalog_B)
    display(iv_B[cols_show].head(30))
    plot_iv(iv_B[iv_B['group'].isin(pkb_groups)], 'Population B: top PKB features by IV')
    catalog_main = catalog
    catalog = pd.concat([catalog, catalog_B[~catalog_B.index.isin(catalog.index)]])
    with Timer('Incremental value, population B'):
        inc_B = incremental_value(
            X_B, y_B, pop_B.set_index('ep_id').loc[X_B.index, 'score_date'].values,
            pop_B.set_index('ep_id').loc[X_B.index, 'iin'].values, list(base_B.columns),
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] in pkb_groups],
            [f for f in F_B.columns if catalog_B.loc[f, 'group'] == 'pkb_own'], 'Population B')
    catalog = catalog_main
else:
    print('Part B skipped')

## 17. Export and summary
Results are written to `OUTPUT_DIR` (Excel when `openpyxl` is installed, otherwise CSV files).
The summary below is generated from the numbers above.

In [ ]:
def bins_long(X, y, models, feats):
    out = []
    for f in feats:
        t = bin_table(X[f], y, models[f])
        t.insert(0, 'feature', f)
        out.append(t)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


sheets = {'iv_population_A': iv_A, 'rule_impact_A': impact_A, 'stability_A': stab_A,
          'bins_A': bins_long(X_A, y_A, models_A, list(iv_A.head(80).index)),
          'cohorts_A': cohorts, 'feature_catalog': catalog, 'own_contract_matching': own_diag}
if inc_A is not None:
    sheets['incremental_A'] = inc_A[0]
if sec:
    sheets['iv_by_target_A'] = iv_targets
if iv_B is not None:
    sheets.update({'iv_population_B': iv_B, 'rule_impact_B': impact_B})
    if inc_B is not None:
        sheets['incremental_B'] = inc_B[0]
try:
    import openpyxl  # noqa: F401
    path = os.path.join(OUTPUT_DIR, 'pkb_monitoring_iv_results.xlsx')
    with pd.ExcelWriter(path) as xw:
        for name, df in sheets.items():
            d = df.copy()
            for c in d.columns:
                if pd.api.types.is_datetime64_any_dtype(d[c]):
                    d[c] = d[c].astype(str)
            d.to_excel(xw, sheet_name=name[:31])
    print('Saved', path)
except ImportError:
    for name, df in sheets.items():
        df.to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), encoding='utf-8-sig')
    print('Saved CSV files to', OUTPUT_DIR)

lines = [f'DATA: {"SYNTHETIC (demo) - numbers say nothing about the real portfolio" if DEMO_MODE else "production"}',
         f'Monitoring coverage used: {HES_START_TS.date()} .. {HES_END_TS.date()}; DPD data until {DATA_END_TS.date()}.',
         f'Population A: {len(pop):,} entries into soft collection scored {pop["score_date"].min().date()} .. '
         f'{pop["score_date"].max().date()}; roll rate to DPD {BAD_DPD}+ = {BR_A:.2%} ({int(y_A.sum()):,} bads).',
         f'{(F_A["days_since_any_oth"].notna()).mean():.0%} of episodes had at least one PKB event at other lenders before scoring.']
pk = iv_A[iv_A['group'].isin(pkb_groups)]
lines.append(f'PKB features: {len(pk)}; IV >= 0.1: {(pk["iv"] >= 0.1).sum()}, 0.02-0.1: {((pk["iv"] >= 0.02) & (pk["iv"] < 0.1)).sum()}, '
             f'below 0.02: {(pk["iv"] < 0.02).sum()}.')
lines.append('Strongest PKB features: ' + '; '.join(f'{f} (IV {r.iv:.3f}, {r.strength})' for f, r in pk.head(5).iterrows()))
top_rules = impact_A[impact_A['reliable'] & (impact_A['group'] != 'pkb_own')].sort_values('iv_flag', ascending=False).head(5)
lines.append('Rules with the largest impact (last %d days): ' % IMPACT_WINDOW + '; '.join(
    f'{r.rule_key} (lift {r.lift:.2f}, {r.share_flagged:.1%} flagged)' for r in top_rules.itertuples()))
if inc_A is not None:
    r = inc_A[0]
    b = r.loc['B: + PKB other lenders'] if 'B: + PKB other lenders' in r.index else None
    if b is not None:
        lines.append(f'Out-of-time Gini: baseline {r.loc["A: baseline", "gini_test"]:.3f} -> with PKB {b["gini_test"]:.3f} '
                     f'(delta {b["delta_gini_vs_A"]:+.3f}, 95% CI {b["delta_CI95"]}).')
susp = pk[pk['iv'] >= 0.5]
if len(susp):
    lines.append('Check for leakage (IV >= 0.5): ' + ', '.join(susp.index[:10]))
if iv_B is not None:
    pb = iv_B[iv_B['group'].isin(pkb_groups)]
    lines.append(f'Part B (early warning): best PKB feature {pb.index[0]} (IV {pb["iv"].iloc[0]:.3f}).')
lines.append(f'Caveat: monitoring history is short ({(HES_END_TS - HES_START_TS).days} days), so {max(WINDOWS)}-day '
             'features of the first weeks see a truncated window and stability covers only a few weekly cohorts.')
print('\n'.join('- ' + s for s in lines))